# Tarea 1 · Contrataciones públicas de la DNCP con Hadoop, Spark, Hive, HBase y Elasticsearch

Universidad Comunera · Maestría en Ciencia de Datos · Análisis y procesamiento de datos masivos
Autores: Bethania Avila, Arturo Fernandez, Gustavo Bogado

Este notebook instala el ecosistema en la máquina de Colab y responde las 7 preguntas de la propuesta.
Cada exportable (CSV en `resultados/`) se calcula con al menos dos herramientas y se compara en el último paso.

| Pregunta | Cálculo principal | Comprobación |
|---|---|---|
| P1 Procesos, adjudicaciones y contratos por año y modalidad | Spark | MapReduce (Hadoop Streaming) y Hive |
| P2 Llamados con una sola oferta | Spark | MapReduce y Hive |
| P3 Desvío entre monto estimado y adjudicado | Spark | Hive |
| P4 Contratos de un proveedor (RUC) en milisegundos | Spark | Elasticsearch y HBase |
| P5 Concentración (top 10 y HHI) | Spark | Hive |
| P6 Plazo de convocatoria y cantidad de oferentes | Spark (MLlib) | Hive y pandas |
| P7 Gasto en combustible, medicamentos y alimentos | Spark | Elasticsearch |

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno` con la RAM más alta disponible.
Con `ALCANCE = "2025"` (182 MB comprimido) el notebook completo tarda unos 30–40 minutos;
`"full"` (histórico 2011–2026, 1,7 GB comprimido, 18 GB descomprimido) puede superar el disco o el tiempo de una sesión gratuita.

In [ ]:
ALCANCE = "2025"   # "2025" para pruebas, "full" para el histórico completo

import os
os.environ.update({
    "ALCANCE": ALCANCE,
    "DNCP_BASE": "/content/dncp",
    "DNCP_JSONL": f"hdfs://localhost:9000/dncp/{ALCANCE}.jsonl",
    "DNCP_PARQUET": f"hdfs://localhost:9000/dncp/parquet_{ALCANCE}",
    "DNCP_PARQUET_LOCAL": f"/content/dncp/parquet_{ALCANCE}",
    "SPARK_MEMORIA": "4g",
    "JAVA_HOME": "/usr/lib/jvm/java-8-openjdk-amd64",
    "HADOOP_HOME": "/opt/hadoop", "HIVE_HOME": "/opt/hive", "HBASE_HOME": "/opt/hbase",
    "HDFS_NAMENODE_USER": "root", "HDFS_DATANODE_USER": "root", "HDFS_SECONDARYNAMENODE_USER": "root",
})
os.environ["PATH"] = ":".join(["/opt/hadoop/bin", "/opt/hadoop/sbin", "/opt/hive/bin", "/opt/hbase/bin",
                               os.environ["PATH"]])
os.makedirs("/content/dncp/datos", exist_ok=True)
%cd /content

## 1. Código del pipeline
Los mismos scripts que se ejecutaron localmente (carpeta `src/` del trabajo).

In [ ]:
!mkdir -p src/mapreduce src/hive && ls -R src

In [ ]:
%%writefile src/comun.py
"""Configuración y utilidades compartidas por todos los scripts del pipeline.

Variables de entorno:
    DNCP_BASE           carpeta de trabajo con los datos (por defecto C:/Users/betha/dncp_bigdata)
    ALCANCE             "full" (histórico 2011-2026) o un año ("2025")
    DNCP_JSONL          ruta del JSON Lines para Spark (en Colab: hdfs://localhost:9000/...)
    DNCP_PARQUET        ruta de las tablas Parquet para Spark/Hive (en Colab: en HDFS)
    DNCP_PARQUET_LOCAL  copia local de las tablas para Elasticsearch, HBase y la verificación
    SPARK_MEMORIA       memoria del driver de Spark (por defecto 12g)
"""
import os
import platform

BASE = os.environ.get("DNCP_BASE", "C:/Users/betha/dncp_bigdata").replace("\\", "/")
ALCANCE = os.environ.get("ALCANCE", "full")

JSONL_GZ = f"{BASE}/datos/{ALCANCE}.jsonl.gz"
# Spark no puede dividir un .gz entre tareas; se trabaja sobre el .jsonl descomprimido
# para que cada bloque de 128 MB sea una tarea, igual que un InputSplit de HDFS.
JSONL = os.environ.get("DNCP_JSONL", f"{BASE}/datos/{ALCANCE}.jsonl")
PARQUET = os.environ.get("DNCP_PARQUET", f"{BASE}/parquet_{ALCANCE}")
PARQUET_LOCAL = os.environ.get("DNCP_PARQUET_LOCAL", PARQUET)

RAIZ = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTADOS = os.path.join(RAIZ, "resultados", ALCANCE)

TABLAS = ["procesos", "adjudicaciones", "contratos", "items_adjudicados"]

# Términos de la pregunta 7 (singular y plural, sin tildes)
TERMINOS_P7 = {
    "combustible": ["combustible", "combustibles"],
    "medicamentos": ["medicamento", "medicamentos"],
    "alimentos": ["alimento", "alimentos"],
}

ES_WINDOWS = platform.system() == "Windows"


def carpeta_resultados(herramienta):
    ruta = os.path.join(RESULTADOS, herramienta)
    os.makedirs(ruta, exist_ok=True)
    return ruta


def spark(app, memoria="12g"):
    """SparkSession local con todos los núcleos."""
    if ES_WINDOWS:
        # Java 23 necesita este permiso para el cliente de Hadoop que trae Spark
        os.environ.setdefault("JAVA_TOOL_OPTIONS", "-Djava.security.manager=allow")
    from pyspark.sql import SparkSession

    return (
        SparkSession.builder.master("local[*]")
        .appName(app)
        .config("spark.driver.memory", os.environ.get("SPARK_MEMORIA", memoria))
        .config("spark.sql.shuffle.partitions", "48")
        .config("spark.sql.session.timeZone", "America/Asuncion")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        # Datos públicos con valores sucios: un campo inválido queda nulo en vez de abortar
        .config("spark.sql.ansi.enabled", "false")
        # ...pero size(null) debe ser nulo y no -1 (comportamiento heredado sin ANSI)
        .config("spark.sql.legacy.sizeOfNull", "false")
        .getOrCreate()
    )


def escribir_parquet(df, ruta, particion=None):
    """Escribe un DataFrame de Spark como carpeta Parquet y devuelve la cantidad de filas.

    En Linux (Colab, clúster) usa el escritor distribuido de Spark. En Windows ese
    escritor exige winutils.exe de Hadoop y los workers de Python fallan con
    Python 3.13, así que el resultado se trae al driver como Arrow (JVM -> Python,
    sin workers) y se guarda con PyArrow. Con `particion` se escribe un archivo por
    valor de esa columna para no traer toda la tabla de una vez.
    """
    if not ES_WINDOWS:
        df.write.mode("overwrite").parquet(ruta)
        return df.sparkSession.read.parquet(ruta).count()
    import shutil

    import pyarrow.parquet as pq
    from pyspark.sql import functions as F

    shutil.rmtree(ruta, ignore_errors=True)
    os.makedirs(ruta)
    if particion is None:
        grupos = [(None, df)]
    else:
        valores = [r[0] for r in df.select(particion).distinct().collect()]
        grupos = [(v, df.where(F.col(particion).eqNullSafe(v))) for v in valores]
    total = 0
    for i, (valor, parte) in enumerate(grupos):
        tabla = parte.toArrow()
        pq.write_table(tabla, f"{ruta}/part-{i:05d}.parquet", compression="snappy")
        total += tabla.num_rows
    return total


def leer_parquet(spark, tabla):
    """Lee una tabla Parquet del pipeline. En Windows se pasan los archivos uno por uno,
    porque listar la carpeta con Hadoop requiere su DLL nativa (hadoop.dll)."""
    ruta = f"{PARQUET}/{tabla}"
    if not ES_WINDOWS:
        return spark.read.parquet(ruta)
    import glob

    archivos = sorted(glob.glob(f"{ruta}/*.parquet"))
    return spark.read.parquet(*[a.replace("\\", "/") for a in archivos])


def guardar_csv(pdf, herramienta, nombre):
    """Guarda un DataFrame de pandas como exportable CSV (UTF-8 con BOM para Excel)."""
    ruta = os.path.join(carpeta_resultados(herramienta), f"{nombre}.csv")
    pdf.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"  -> {os.path.relpath(ruta, RAIZ)} ({len(pdf):,} filas)")
    return ruta

In [ ]:
%%writefile src/01_aplanar_spark.py
"""Paso 1 (Apache Spark): aplanar el JSON Lines OCDS de la DNCP en cuatro tablas Parquet.

Cada línea del archivo es un proceso de contratación (ocid) con sus etapas anidadas.
Se leen solo los campos necesarios con un esquema explícito y se generan:

    procesos           una fila por proceso (llamado)
    adjudicaciones     una fila por adjudicación y proveedor
    contratos          una fila por contrato y proveedor
    items_adjudicados  una fila por ítem de adjudicación válida

Uso:  python src/01_aplanar_spark.py
"""
import time

from pyspark.sql import functions as F

import comun

ESQUEMA = """
ocid STRING,
date STRING,
buyer STRUCT<id: STRING, name: STRING>,
parties ARRAY<STRUCT<id: STRING, roles: ARRAY<STRING>,
                     details: STRUCT<level: STRING, entityType: STRING>>>,
tender STRUCT<
    status: STRING,
    datePublished: STRING,
    procurementMethod: STRING,
    procurementMethodDetails: STRING,
    mainProcurementCategoryDetails: STRING,
    value: STRUCT<amount: DOUBLE, currency: STRING>,
    numberOfTenderers: BIGINT,
    tenderers: ARRAY<STRUCT<id: STRING>>,
    tenderPeriod: STRUCT<startDate: STRING, endDate: STRING>>,
bids STRUCT<details: ARRAY<STRUCT<tenderers: ARRAY<STRUCT<id: STRING>>>>>,
awards ARRAY<STRUCT<
    id: STRING, status: STRING, date: STRING,
    value: STRUCT<amount: DOUBLE, currency: STRING>,
    suppliers: ARRAY<STRUCT<id: STRING, name: STRING>>,
    items: ARRAY<STRUCT<
        id: STRING, description: STRING, quantity: DOUBLE,
        unit: STRUCT<value: STRUCT<amount: DOUBLE, currency: STRING>>,
        classification: STRUCT<id: STRING, description: STRING>>>>>,
contracts ARRAY<STRUCT<
    id: STRING, awardID: STRING, status: STRING, dateSigned: STRING,
    value: STRUCT<amount: DOUBLE, currency: STRING>,
    period: STRUCT<startDate: STRING>>>,
_corrupt_record STRING
"""

ESTADOS_VALIDOS = ["active", "closed"]  # adjudicaciones vigentes o finalizadas


def limpiar(col):
    """Espacios, tabuladores y saltos de línea seguidos -> un espacio; vacío -> nulo."""
    texto = F.trim(F.regexp_replace(col, r"\s+", " "))
    return F.when(texto != "", texto)


def ruc(col):
    """PY-RUC-80000519-8 -> 80000519-8 (otros esquemas quedan como vienen)."""
    return F.regexp_replace(col, "^PY-RUC-", "")


def main():
    spark = comun.spark("01_aplanar")
    inicio = time.time()

    crudo = (spark.read.schema(ESQUEMA)
             .option("mode", "PERMISSIVE")
             .option("columnNameOfCorruptRecord", "_corrupt_record")
             .json(comun.JSONL))

    t = F.col("tender")
    fecha_ref = F.coalesce(t.tenderPeriod.startDate, t.datePublished, F.col("date"))
    oferentes_bids = F.array_distinct(F.flatten(F.transform("bids.details", lambda d: d.tenderers.id)))
    comprador = F.filter("parties", lambda p: F.array_contains(p.roles, "buyer"))

    base = crudo.select(
        "ocid",
        F.substring(fecha_ref, 1, 4).cast("int").alias("anio"),
        F.to_date(F.substring(t.tenderPeriod.startDate, 1, 10)).alias("fecha_inicio"),
        F.to_date(F.substring(t.tenderPeriod.endDate, 1, 10)).alias("fecha_fin"),
        F.lower(limpiar(t.procurementMethodDetails)).alias("modalidad"),
        t.procurementMethod.alias("metodo"),
        limpiar(t.mainProcurementCategoryDetails).alias("categoria"),
        F.col("buyer.id").alias("entidad_id"),
        limpiar(F.col("buyer.name")).alias("entidad"),
        F.coalesce(limpiar(F.element_at(comprador, 1).details.level), F.lit("Sin dato")).alias("tipo_entidad"),
        t.status.alias("estado_llamado"),
        t.value.currency.alias("moneda_estimado"),
        t.value.amount.alias("monto_estimado"),
        F.greatest(F.coalesce(t.numberOfTenderers, F.lit(0)),
                   F.coalesce(F.size(t.tenderers), F.lit(0)),
                   F.coalesce(F.size(oferentes_bids), F.lit(0))).alias("n_oferentes"),
        "awards", "contracts", "_corrupt_record",
        F.col("tender").isNull().alias("sin_llamado"),
    ).persist()

    corruptos = base.where(F.col("_corrupt_record").isNotNull()).count()
    sin_llamado = base.where(F.col("sin_llamado") & F.col("_corrupt_record").isNull()).count()
    base = base.where(F.col("_corrupt_record").isNull() & ~F.col("sin_llamado"))                .drop("_corrupt_record", "sin_llamado")

    # Tipo de entidad: hay registros que no lo traen en `parties`. Se toma el informado en
    # otros procesos de la misma entidad (ninguna entidad tiene dos tipos distintos).
    tipos = (base.where(F.col("tipo_entidad") != "Sin dato").groupBy("entidad_id")
             .agg(F.max("tipo_entidad").alias("tipo_canonico")))
    base = base.join(F.broadcast(tipos), "entidad_id", "left")
    imputados = base.where((F.col("tipo_entidad") == "Sin dato") & F.col("tipo_canonico").isNotNull()).count()
    base = (base.withColumn("tipo_entidad", F.coalesce("tipo_canonico", "tipo_entidad"))
            .drop("tipo_canonico"))
    atributos = ["ocid", "anio", "modalidad", "categoria", "entidad_id", "entidad", "tipo_entidad"]

    # --- adjudicaciones (adjudicación x proveedor) ---
    adj = (base.select(*atributos, F.explode("awards").alias("a"))
           .select(*atributos,
                   F.col("a.id").alias("adjudicacion_id"),
                   F.col("a.status").alias("estado"),
                   F.col("a.status").isin(ESTADOS_VALIDOS).alias("valida"),
                   F.to_date(F.substring("a.date", 1, 10)).alias("fecha"),
                   F.col("a.value.amount").alias("monto"),
                   F.col("a.value.currency").alias("moneda"),
                   F.coalesce(F.size("a.suppliers"), F.lit(0)).alias("n_proveedores"),
                   F.explode_outer("a.suppliers").alias("s"))
           .select("*", ruc(F.col("s.id")).alias("ruc"), limpiar(F.col("s.name")).alias("proveedor"))
           .withColumn("monto_atribuido", F.col("monto") / F.greatest(F.col("n_proveedores"), F.lit(1)))
           .drop("s"))

    # --- contratos (contrato x proveedor, el proveedor sale de la adjudicación) ---
    proveedores_adj = adj.select("ocid", "adjudicacion_id", "ruc", "proveedor").distinct()
    contratos = (base.select(*atributos, F.explode("contracts").alias("c"))
                 .where(F.col("c.awardID").isNotNull())
                 .select(*atributos,
                         F.col("c.id").alias("contrato_id"),
                         F.col("c.awardID").alias("adjudicacion_id"),
                         F.col("c.status").alias("estado"),
                         F.to_date(F.substring(F.coalesce("c.dateSigned", "c.period.startDate"), 1, 10))
                          .alias("fecha_firma"),
                         F.col("c.value.amount").alias("monto"),
                         F.col("c.value.currency").alias("moneda"))
                 .join(proveedores_adj, ["ocid", "adjudicacion_id"], "left"))

    # --- ítems de adjudicaciones válidas ---
    items = (base.select(*atributos, F.explode("awards").alias("a"))
             .where(F.col("a.status").isin(ESTADOS_VALIDOS))
             .select(*atributos, F.col("a.id").alias("adjudicacion_id"), F.explode("a.items").alias("i"))
             .select(*atributos, "adjudicacion_id",
                     F.col("i.id").alias("item_id"),
                     F.col("i.description").alias("descripcion"),
                     F.col("i.classification.id").alias("catalogo_id"),
                     F.col("i.classification.description").alias("catalogo"),
                     F.col("i.quantity").alias("cantidad"),
                     F.col("i.unit.value.amount").alias("precio_unitario"),
                     F.col("i.unit.value.currency").alias("moneda"))
             .withColumn("monto", F.col("cantidad") * F.col("precio_unitario")))

    # --- procesos (una fila por llamado) ---
    adj_validas = F.filter("awards", lambda a: a.status.isin(ESTADOS_VALIDOS))
    adj_pyg = F.filter(adj_validas, lambda a: a.value.currency == "PYG")
    procesos = base.select(
        "ocid", "anio", "fecha_inicio", "fecha_fin",
        F.datediff("fecha_fin", "fecha_inicio").alias("dias_convocatoria"),
        "modalidad", "metodo", "categoria", "entidad_id", "entidad", "tipo_entidad",
        "estado_llamado", "moneda_estimado", "monto_estimado", "n_oferentes",
        F.coalesce(F.size(adj_validas), F.lit(0)).alias("n_adjudicaciones"),
        F.coalesce(F.size(F.filter("contracts", lambda c: c.awardID.isNotNull())), F.lit(0))
         .alias("n_contratos"),
        F.coalesce(F.aggregate(adj_pyg, F.lit(0.0),
                               lambda acc, a: acc + F.coalesce(a.value.amount, F.lit(0.0))), F.lit(0.0))
         .alias("monto_adjudicado_pyg"),
        F.coalesce(F.size(adj_pyg), F.lit(0)).alias("n_adjudicaciones_pyg"),
    )

    filas = {}
    for nombre, df in [("procesos", procesos), ("adjudicaciones", adj),
                       ("contratos", contratos), ("items_adjudicados", items)]:
        t0 = time.time()
        filas[nombre] = comun.escribir_parquet(df, f"{comun.PARQUET}/{nombre}", particion="anio")
        print(f"{nombre}: {filas[nombre]:,} filas ({time.time() - t0:.0f} s)")

    import pandas as pd
    resumen = pd.DataFrame([{"tabla": k, "filas": v} for k, v in filas.items()]
                           + [{"tabla": "registros_corruptos", "filas": corruptos},
                              {"tabla": "procesos_sin_llamado", "filas": sin_llamado},
                              {"tabla": "procesos_tipo_entidad_imputado", "filas": imputados}])
    comun.guardar_csv(resumen, "spark", "00_tablas_aplanadas")
    print(f"Total: {time.time() - inicio:.0f} s")
    spark.stop()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/02_preguntas_spark.py
"""Paso 2 (Apache Spark): responder las 7 preguntas sobre las tablas Parquet.

Cada exportable se guarda en resultados/<alcance>/spark/ y luego se compara con
MapReduce, Hive y Elasticsearch en src/06_verificar.py.

Uso:  python src/02_preguntas_spark.py
"""
import time

import pandas as pd
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import Window
from pyspark.sql import functions as F

import comun

SIN_DATO = "(sin dato)"
MIN_ADJ_ENTIDAD = 20      # entidades con menos adjudicaciones no se clasifican por HHI
MIN_PROCESOS_CATEGORIA = 30
TRAMOS_DIAS = [(0, 7, "0-7"), (8, 14, "08-14"), (15, 21, "15-21"), (22, 30, "22-30"),
               (31, 60, "31-60"), (61, 365, "61-365")]

# P7 por catálogo: los 8 primeros dígitos del código DNCP son el código UNSPSC
PREFIJOS_UNSPSC = {
    "combustible": "1510",   # Combustibles: petróleo y destilados (gasoil, nafta, kerosene)
    "medicamentos": "51",    # Medicamentos y productos farmacéuticos
    "alimentos": "50",       # Alimentos, bebidas y tabaco
}


def texto(col):
    return F.coalesce(col.cast("string"), F.lit(SIN_DATO))


def guardar(df, nombre):
    return comun.guardar_csv(df.toPandas(), "spark", nombre)


def mediana_y_cuartiles(col):
    return [F.percentile(col, 0.25).alias("p25"), F.percentile(col, 0.5).alias("mediana"),
            F.percentile(col, 0.75).alias("p75")]


def normalizar(col):
    """minúsculas y sin tildes, para buscar palabras completas."""
    return F.translate(F.lower(col), "áéíóúüñàèìòù", "aeiouunaeiou")


def p1(procesos):
    print("P1: procesos, adjudicaciones y contratos por año y modalidad")
    df = (procesos.groupBy(texto(F.col("anio")).alias("anio"), texto(F.col("modalidad")).alias("modalidad"))
          .agg(F.count("*").alias("procesos"),
               F.sum("n_adjudicaciones").alias("adjudicaciones"),
               F.sum("n_contratos").alias("contratos"),
               F.sum("monto_adjudicado_pyg").alias("monto_adjudicado_pyg"))
          .orderBy("anio", "modalidad"))
    guardar(df, "p1_anio_modalidad")
    guardar(df.groupBy("anio").agg(*[F.sum(c).alias(c) for c in
                                     ["procesos", "adjudicaciones", "contratos", "monto_adjudicado_pyg"]])
            .orderBy("anio"), "p1_anio")


def p2(procesos):
    print("P2: llamados con una sola oferta")
    con_oferta = procesos.where(F.col("n_oferentes") >= 1)
    for dim, nombre in [("modalidad", "p2_unica_oferta_modalidad"),
                        ("tipo_entidad", "p2_unica_oferta_tipo_entidad"),
                        ("anio", "p2_unica_oferta_anio")]:
        df = (con_oferta.groupBy(texto(F.col(dim)).alias(dim))
              .agg(F.count("*").alias("llamados_con_oferta"),
                   F.sum((F.col("n_oferentes") == 1).cast("int")).alias("una_sola_oferta"))
              .withColumn("pct_una_oferta",
                          F.round(100 * F.col("una_sola_oferta") / F.col("llamados_con_oferta"), 2))
              .orderBy(dim))
        guardar(df, nombre)


def p3(procesos):
    print("P3: desvío entre monto estimado y adjudicado")
    base = (procesos
            .where((F.col("moneda_estimado") == "PYG") & (F.col("monto_estimado") > 0)
                   & (F.col("n_adjudicaciones_pyg") > 0) & (F.col("monto_adjudicado_pyg") > 0))
            .withColumn("desvio_pct", 100 * (F.col("monto_adjudicado_pyg") - F.col("monto_estimado"))
                        / F.col("monto_estimado")))

    def resumen(df):
        return df.agg(
            F.count("*").alias("procesos"),
            *mediana_y_cuartiles(F.col("desvio_pct")),
            F.percentile("desvio_pct", 0.9).alias("p90"),
            F.round(100 * F.avg((F.col("desvio_pct") > 0).cast("int")), 2).alias("pct_sobre_estimado"),
            F.round(100 * F.avg((F.col("desvio_pct") > 10).cast("int")), 2).alias("pct_sobre_10"),
            F.round(100 * F.avg((F.col("desvio_pct") < -10).cast("int")), 2).alias("pct_bajo_10"),
            F.sum("monto_estimado").alias("monto_estimado_pyg"),
            F.sum("monto_adjudicado_pyg").alias("monto_adjudicado_pyg"))

    guardar(resumen(base), "p3_desvio_resumen")
    for dim, nombre in [("categoria", "p3_desvio_categoria"), ("modalidad", "p3_desvio_modalidad")]:
        df = (resumen(base.groupBy(texto(F.col(dim)).alias(dim)))
              .withColumn("desvio_agregado_pct",
                          F.round(100 * (F.col("monto_adjudicado_pyg") - F.col("monto_estimado_pyg"))
                                  / F.col("monto_estimado_pyg"), 2))
              .where(F.col("procesos") >= MIN_PROCESOS_CATEGORIA)
              .orderBy(F.desc("pct_sobre_10")))
        guardar(df, nombre)


def p4(contratos):
    print("P4: contratos por proveedor (RUC)")
    pyg = contratos.where((F.col("moneda") == "PYG") & F.col("ruc").isNotNull())
    top = (pyg.groupBy("ruc")
           .agg(F.max("proveedor").alias("proveedor"),
                F.count("*").alias("contratos"),
                F.countDistinct("entidad_id").alias("entidades"),
                F.sum("monto").alias("monto_pyg"))
           .orderBy(F.desc("monto_pyg")))
    top.cache()
    guardar(top.limit(50), "p4_top_proveedores")

    # Muestra de RUC para comparar la latencia de Spark con Elasticsearch y HBase
    por_monto = [r.ruc for r in top.limit(5).collect()]
    por_cantidad = [r.ruc for r in top.orderBy(F.desc("contratos"), "ruc").limit(5).collect()]
    al_azar = [r.ruc for r in top.where(F.col("contratos") >= 3)
               .orderBy(F.xxhash64("ruc"), "ruc").limit(5).collect()]
    muestra = list(dict.fromkeys(por_monto + por_cantidad + al_azar))
    comun.guardar_csv(pd.DataFrame({"ruc": muestra}), "spark", "p4_rucs_muestra")

    filas = []
    for ruc in muestra:
        t0 = time.perf_counter()
        r = (contratos.where(F.col("ruc") == ruc)
             .agg(F.count("*").alias("contratos"),
                  F.countDistinct("entidad_id").alias("entidades"),
                  F.sum(F.when(F.col("moneda") == "PYG", F.col("monto"))).alias("monto_pyg"))
             .collect()[0])
        filas.append({"ruc": ruc, "contratos": r.contratos, "entidades": r.entidades,
                      "monto_pyg": r.monto_pyg, "ms_spark": round(1000 * (time.perf_counter() - t0), 1)})
    comun.guardar_csv(pd.DataFrame(filas), "spark", "p4_consulta_ruc")

    ejemplo = muestra[0]
    guardar(contratos.where(F.col("ruc") == ejemplo)
            .select("ruc", "proveedor", "ocid", "contrato_id", "fecha_firma", "entidad", "tipo_entidad",
                    "modalidad", "categoria", "monto", "moneda")
            .orderBy(F.desc("monto")), "p4_ejemplo_contratos_ruc")


def concentracion(adj, claves):
    """Participación del top 1 y top 10 de proveedores e índice Herfindahl-Hirschman."""
    por_prov = adj.groupBy(*claves, "ruc").agg(F.sum("monto_atribuido").alias("monto"),
                                               F.count("*").alias("n"))
    w_total = Window.partitionBy(*claves)
    w_rank = Window.partitionBy(*claves).orderBy(F.desc("monto"), "ruc")
    con_share = (por_prov.withColumn("total", F.sum("monto").over(w_total))
                 .withColumn("share", 100 * F.col("monto") / F.col("total"))
                 .withColumn("rango", F.row_number().over(w_rank)))
    return (con_share.groupBy(*claves)
            .agg(F.sum("n").alias("adjudicaciones"),
                 F.count("*").alias("proveedores"),
                 F.first("total").alias("monto_pyg"),
                 F.round(F.sum(F.when(F.col("rango") == 1, F.col("share"))), 4).alias("share_top1"),
                 F.round(F.sum(F.when(F.col("rango") <= 10, F.col("share"))), 4).alias("share_top10"),
                 F.round(F.sum(F.col("share") * F.col("share")), 4).alias("hhi")))


def nivel_hhi(col):
    return (F.when(col < 1500, "1. no concentrado (<1500)")
            .when(col <= 2500, "2. moderado (1500-2500)")
            .otherwise("3. alto (>2500)"))


def p5(adjudicaciones):
    print("P5: concentración de proveedores (top 10 y HHI)")
    adj = adjudicaciones.where(F.col("valida") & (F.col("moneda") == "PYG")
                               & (F.col("monto_atribuido") > 0) & F.col("ruc").isNotNull()
                               & F.col("entidad_id").isNotNull())
    # Se agrupa por entidad_id: el nombre de una entidad cambia entre años
    nombres = adj.groupBy("entidad_id").agg(F.max("entidad").alias("entidad"),
                                            F.max("tipo_entidad").alias("tipo_entidad"))
    entidades = (concentracion(adj, ["entidad_id"]).join(nombres, "entidad_id")
                 .select("entidad_id", "entidad", "tipo_entidad", "adjudicaciones", "proveedores", "monto_pyg",
                         "share_top1", "share_top10", "hhi")
                 .orderBy(F.desc("monto_pyg")))
    entidades.cache()
    guardar(entidades, "p5_concentracion_entidad")
    guardar(concentracion(adj.withColumn("categoria", texto(F.col("categoria"))), ["categoria"])
            .orderBy(F.desc("hhi")), "p5_concentracion_categoria")

    clasificables = entidades.where(F.col("adjudicaciones") >= MIN_ADJ_ENTIDAD)
    guardar(clasificables.groupBy(nivel_hhi(F.col("hhi")).alias("nivel_hhi"))
            .agg(F.count("*").alias("entidades"), F.sum("monto_pyg").alias("monto_pyg"))
            .orderBy("nivel_hhi"), "p5_entidades_por_nivel_hhi")
    guardar(clasificables.groupBy("tipo_entidad")
            .agg(F.count("*").alias("entidades"),
                 F.percentile("hhi", 0.5).alias("hhi_mediana"),
                 F.percentile("share_top10", 0.5).alias("share_top10_mediana"))
            .orderBy(F.desc("hhi_mediana")), "p5_hhi_tipo_entidad")

    total = adj.agg(F.sum("monto_atribuido")).collect()[0][0]
    guardar(adj.groupBy("ruc").agg(F.max("proveedor").alias("proveedor"),
                                   F.sum("monto_atribuido").alias("monto_pyg"))
            .withColumn("share_pct", F.round(100 * F.col("monto_pyg") / F.lit(total), 4))
            .orderBy(F.desc("monto_pyg")).limit(10), "p5_top10_proveedores_estado")


def p6(procesos):
    print("P6: duración de la convocatoria y cantidad de oferentes")
    validos = procesos.where(F.col("dias_convocatoria").between(0, 365))
    guardar(procesos.agg(
        F.count("*").alias("procesos"),
        F.sum(F.col("dias_convocatoria").isNull().cast("int")).alias("sin_fechas"),
        F.sum((F.col("dias_convocatoria") < 0).cast("int")).alias("dias_negativos"),
        F.sum((F.col("dias_convocatoria") > 365).cast("int")).alias("mas_de_365")), "p6_calidad_fechas")

    guardar(validos.groupBy(texto(F.col("modalidad")).alias("modalidad"))
            .agg(F.count("*").alias("procesos"), F.round(F.avg("dias_convocatoria"), 2).alias("media"),
                 *mediana_y_cuartiles(F.col("dias_convocatoria")))
            .where(F.col("procesos") >= MIN_PROCESOS_CATEGORIA)
            .orderBy(F.desc("procesos")), "p6_dias_modalidad")

    con_oferta = validos.where(F.col("n_oferentes") >= 1)
    tramo = F.lit(None)
    for desde, hasta, etiqueta in reversed(TRAMOS_DIAS):
        tramo = F.when(F.col("dias_convocatoria").between(desde, hasta), etiqueta).otherwise(tramo)
    guardar(con_oferta.groupBy(tramo.alias("tramo_dias"))
            .agg(F.count("*").alias("llamados"),
                 F.round(F.avg("n_oferentes"), 4).alias("oferentes_promedio"),
                 F.percentile("n_oferentes", 0.5).alias("oferentes_mediana"),
                 F.round(100 * F.avg((F.col("n_oferentes") == 1).cast("int")), 2).alias("pct_una_oferta"))
            .orderBy("tramo_dias"), "p6_tramos_dias_oferentes")

    # Spearman con MLlib: global y dentro de las modalidades más frecuentes
    vector = VectorAssembler(inputCols=["dias_convocatoria", "n_oferentes"], outputCol="v")
    grupos = [("(todas)", con_oferta)] + [
        (r.modalidad, con_oferta.where(F.col("modalidad") == r.modalidad))
        for r in con_oferta.groupBy("modalidad").count().orderBy(F.desc("count")).limit(6).collect()]
    filas = []
    for nombre, df in grupos:
        d = df.select(F.col("dias_convocatoria").cast("double"), F.col("n_oferentes").cast("double"))
        rho = Correlation.corr(vector.transform(d), "v", "spearman").collect()[0][0][0, 1]
        filas.append({"modalidad": nombre, "llamados": d.count(), "spearman": round(float(rho), 4)})
    comun.guardar_csv(pd.DataFrame(filas), "spark", "p6_spearman")


def p7(items):
    print("P7: gasto en combustible, medicamentos y alimentos")
    base = items.where((F.col("moneda") == "PYG") & F.col("monto").isNotNull())
    desc = normalizar(F.col("descripcion"))
    marcados = base
    for grupo, palabras in comun.TERMINOS_P7.items():
        # (?U): límites de palabra Unicode, como Elasticsearch (desde Java 19, \b es solo ASCII)
        patron = r"(?U)\b(" + "|".join(palabras) + r")\b"
        marcados = marcados.withColumn(f"txt_{grupo}", F.coalesce(desc.rlike(patron), F.lit(False)))
        marcados = marcados.withColumn(f"cat_{grupo}", F.coalesce(
            F.col("catalogo_id").startswith(PREFIJOS_UNSPSC[grupo]), F.lit(False)))
    marcados.cache()

    for metodo, sufijo in [("txt", "texto"), ("cat", "catalogo")]:
        resumen, entidades = [], []
        for grupo in comun.TERMINOS_P7:
            sel = marcados.where(F.col(f"{metodo}_{grupo}"))
            r = sel.agg(F.count("*").alias("items"), F.countDistinct("ocid").alias("procesos"),
                        F.sum("monto").alias("monto_pyg")).collect()[0]
            resumen.append({"grupo": grupo, "items": r.items, "procesos": r.procesos, "monto_pyg": r.monto_pyg})
            top = (sel.where(F.col("entidad").isNotNull())
                   .groupBy("entidad").agg(F.sum("monto").alias("monto_pyg"), F.count("*").alias("items"))
                   .orderBy(F.desc("monto_pyg"), "entidad").limit(10).toPandas())
            top.insert(0, "grupo", grupo)
            entidades.append(top)
        comun.guardar_csv(pd.DataFrame(resumen), "spark", f"p7_{sufijo}_resumen")
        comun.guardar_csv(pd.concat(entidades), "spark", f"p7_{sufijo}_top_entidades")

    # ¿Cuántos ítems hallados por texto son realmente de la familia del catálogo?
    filas = []
    for grupo in comun.TERMINOS_P7:
        r = marcados.agg(
            F.sum(F.col(f"txt_{grupo}").cast("int")).alias("solo_texto_total"),
            F.sum((F.col(f"txt_{grupo}") & F.col(f"cat_{grupo}")).cast("int")).alias("texto_y_catalogo"),
            F.sum(F.col(f"cat_{grupo}").cast("int")).alias("catalogo_total")).collect()[0]
        filas.append({"grupo": grupo, "items_texto": r.solo_texto_total, "items_catalogo": r.catalogo_total,
                      "items_en_ambos": r.texto_y_catalogo})
    comun.guardar_csv(pd.DataFrame(filas), "spark", "p7_texto_vs_catalogo")

    # Ejemplos de lo que encuentra la búsqueda por texto, para el informe
    ejemplos = (marcados.where(F.col("txt_combustible"))
                .groupBy(F.coalesce("catalogo", F.lit(SIN_DATO)).alias("catalogo"))
                .agg(F.count("*").alias("items"), F.sum("monto").alias("monto_pyg"))
                .orderBy(F.desc("items")).limit(15))
    guardar(ejemplos, "p7_texto_combustible_por_catalogo")


def main():
    spark = comun.spark("02_preguntas")
    t = {n: comun.leer_parquet(spark, n) for n in comun.TABLAS}
    inicio = time.time()
    for paso, args in [(p1, ["procesos"]), (p2, ["procesos"]), (p3, ["procesos"]), (p4, ["contratos"]),
                       (p5, ["adjudicaciones"]), (p6, ["procesos"]), (p7, ["items_adjudicados"])]:
        t0 = time.time()
        paso(*[t[a] for a in args])
        print(f"   {time.time() - t0:.0f} s")
    print(f"Total: {time.time() - inicio:.0f} s")
    spark.stop()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/03_mapreduce.py
"""Paso 3 (MapReduce): preguntas 1 y 2 con el mapper y reducer de Hadoop Streaming.

Dos modos:

  python src/03_mapreduce.py
      Ejecuta el mismo flujo que Hadoop Streaming en esta máquina: divide el JSON
      Lines en splits alineados a líneas (como los InputSplit de 128 MB de HDFS),
      corre un proceso mapper por split en paralelo, ordena la salida por clave
      (shuffle & sort) y la pasa por el reducer. Son dos trabajos encadenados:
        1. mapper_entidades.py / reducer_entidades.py: tabla entidad -> tipo
        2. mapper.py / reducer.py: P1 y P2, con la tabla anterior como caché

  python src/03_mapreduce.py --hadoop salida_hadoop.txt
      Convierte la salida real de `hadoop jar hadoop-streaming.jar` (por ejemplo,
      `hdfs dfs -getmerge /dncp/salida_mr salida_hadoop.txt` en Colab) a los CSV.
"""
import argparse
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

import comun

CARPETA = os.path.join(os.path.dirname(os.path.abspath(__file__)), "mapreduce")
TAM_SPLIT = 128 * 1024 * 1024


def splits(ruta):
    """Rangos de bytes de ~128 MB que terminan en fin de línea."""
    tam = os.path.getsize(ruta)
    rangos, inicio = [], 0
    with open(ruta, "rb") as f:
        while inicio < tam:
            f.seek(min(inicio + TAM_SPLIT, tam))
            f.readline()
            fin = min(f.tell(), tam)
            rangos.append((inicio, fin))
            inicio = fin
    return rangos


def tarea_map(args):
    ruta, (inicio, fin), mapper, argumentos = args
    proc = subprocess.Popen([sys.executable, os.path.join(CARPETA, mapper), *argumentos],
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE)

    def alimentar():
        with open(ruta, "rb") as f:
            f.seek(inicio)
            restante = fin - inicio
            while restante > 0:
                bloque = f.read(min(8 * 1024 * 1024, restante))
                proc.stdin.write(bloque)
                restante -= len(bloque)
        proc.stdin.close()

    with ThreadPoolExecutor(1) as ex:
        ex.submit(alimentar)
        salida = proc.stdout.read()
    if proc.wait() != 0:
        raise RuntimeError(f"el mapper falló en el split {inicio}-{fin}")
    return salida


def ordenar_y_reducir(lineas, reducer):
    """Ordena por los bytes de la clave (como Hadoop) y pasa las líneas por el reducer."""
    lineas = sorted(lineas, key=lambda l: l.encode("utf-8"))
    salida = subprocess.run([sys.executable, os.path.join(CARPETA, reducer)],
                            input="".join(lineas).encode("utf-8"), capture_output=True, check=True)
    return salida.stdout.decode("utf-8").replace("\r\n", "\n").splitlines(keepends=True)


CONTADORES = []


def trabajo(nombre, mapper, reducer, argumentos=()):
    """Un trabajo de Hadoop Streaming: map en paralelo por split, combiner, shuffle & sort, reduce.

    Como en el laboratorio de la clase 3, el reducer también sirve de combiner porque sus operaciones
    (suma y máximo) son asociativas: cada split se resume antes del shuffle y viajan menos registros.
    """
    rangos = splits(comun.JSONL)
    t0 = time.time()
    with ThreadPoolExecutor(os.cpu_count()) as ex:
        salidas = list(ex.map(tarea_map, [(comun.JSONL, r, mapper, list(argumentos)) for r in rangos]))
        por_split = [s.decode("utf-8").splitlines(keepends=True) for s in salidas]
        combinadas = list(ex.map(lambda lineas: ordenar_y_reducir(lineas, reducer), por_split))
    salida_map = sum(len(l) for l in por_split)
    salida_combiner = sum(len(l) for l in combinadas)
    reducido = ordenar_y_reducir([l for c in combinadas for l in c], reducer)
    CONTADORES.append({"trabajo": nombre, "splits": len(rangos), "map_output_records": salida_map,
                       "combine_output_records": salida_combiner, "reduce_output_records": len(reducido),
                       "segundos": round(time.time() - t0, 1)})
    print(f"{nombre}: {len(rangos)} splits, {os.cpu_count()} mappers en paralelo · map {salida_map:,} pares · "
          f"combiner {salida_combiner:,} · reduce {len(reducido):,} · {time.time() - t0:.0f} s")
    return "".join(reducido)


def ejecutar_local():
    tipos = trabajo("trabajo 1 (entidad -> tipo)", "mapper_entidades.py", "reducer_entidades.py")
    cache = os.path.join(comun.BASE, f"entidades_tipo_{comun.ALCANCE}.tsv")
    with open(cache, "w", encoding="utf-8", newline="\n") as f:
        f.write(tipos.replace("\r\n", "\n"))
    print(f"  caché distribuida: {len(tipos.splitlines()):,} entidades con tipo")
    return trabajo("trabajo 2 (P1 y P2)", "mapper.py", "reducer.py", [cache])


def a_csv(texto):
    filas = [l.split("\t") for l in texto.splitlines() if l.strip()]
    p1 = pd.DataFrame([f[1:] for f in filas if f[0] == "P1"],
                      columns=["anio", "modalidad", "procesos", "adjudicaciones", "contratos",
                               "monto_adjudicado_pyg"])
    for c in ["procesos", "adjudicaciones", "contratos"]:
        p1[c] = p1[c].astype(int)
    p1["monto_adjudicado_pyg"] = p1["monto_adjudicado_pyg"].astype(float)
    comun.guardar_csv(p1.sort_values(["anio", "modalidad"]), "mapreduce", "p1_anio_modalidad")

    for codigo, dim, nombre in [("P2M", "modalidad", "p2_unica_oferta_modalidad"),
                                ("P2E", "tipo_entidad", "p2_unica_oferta_tipo_entidad"),
                                ("P2A", "anio", "p2_unica_oferta_anio")]:
        df = pd.DataFrame([f[1:] for f in filas if f[0] == codigo],
                          columns=[dim, "llamados_con_oferta", "una_sola_oferta"])
        df[["llamados_con_oferta", "una_sola_oferta"]] = df[["llamados_con_oferta",
                                                             "una_sola_oferta"]].astype(int)
        df["pct_una_oferta"] = (100 * df["una_sola_oferta"] / df["llamados_con_oferta"]).round(2)
        comun.guardar_csv(df.sort_values(dim), "mapreduce", nombre)

    errores = sum(int(f[2]) for f in filas if f[0] == "ERR")
    print(f"líneas con JSON inválido: {errores}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--hadoop", help="archivo con la salida de Hadoop Streaming")
    args = parser.parse_args()
    if args.hadoop:
        with open(args.hadoop, encoding="utf-8") as f:
            texto = f.read()
    else:
        texto = ejecutar_local()
        comun.guardar_csv(pd.DataFrame(CONTADORES), "mapreduce", "00_contadores")
    a_csv(texto)

In [ ]:
%%writefile src/04_hive.py
"""Paso 4 (Apache Hive): ejecuta src/hive/preguntas.hql y guarda cada exportable.

  python src/04_hive.py --beeline "jdbc:hive2://"
      Ejecuta el HiveQL en Apache Hive real (Colab: Hive 3.1 sobre HDFS) con beeline.

  python src/04_hive.py
      Sin Hive instalado (Windows), ejecuta el mismo HiveQL con el motor SQL de Spark
      (compatible con HiveQL). Las tablas externas se reemplazan por vistas temporales
      sobre el mismo Parquet; las consultas SELECT se ejecutan sin cambios.
"""
import argparse
import io
import re
import subprocess
import time

import pandas as pd

import comun

HQL = f"{comun.RAIZ}/src/hive/preguntas.hql"


def sentencias(ruta, parquet):
    """Divide el script en sentencias y asocia cada SELECT con su nombre de exportable."""
    texto = open(ruta, encoding="utf-8").read().replace("${PARQUET}", parquet)
    resultado, nombre, actual = [], None, []
    for linea in texto.splitlines():
        marca = re.match(r"\s*--\s*exportable:\s*(\S+)", linea)
        if marca:
            nombre = marca.group(1)
            continue
        if linea.strip().startswith("--"):
            continue
        actual.append(linea)
        if linea.rstrip().endswith(";"):
            sql = "\n".join(actual).strip().rstrip(";").strip()
            if sql:
                resultado.append((nombre, sql))
            nombre, actual = None, []
    return resultado


def con_spark():
    spark = comun.spark("04_hive_sql")
    for tabla in ["procesos", "adjudicaciones"]:
        comun.leer_parquet(spark, tabla).createOrReplaceTempView(tabla)
    for nombre, sql in sentencias(HQL, comun.PARQUET):
        inicial = sql.split()[0].upper()
        if inicial in ("USE", "DROP") or sql.upper().startswith(("CREATE DATABASE", "CREATE EXTERNAL")):
            continue  # el catálogo de Hive se reemplaza por vistas temporales
        if sql.upper().startswith("CREATE VIEW"):
            sql = re.sub(r"(?i)^CREATE VIEW", "CREATE OR REPLACE TEMPORARY VIEW", sql)
        t0 = time.time()
        df = spark.sql(sql)
        if nombre:
            comun.guardar_csv(df.toPandas(), "hive", nombre)
            print(f"   {time.time() - t0:.1f} s")
    spark.stop()


def con_beeline(url, parquet):
    def beeline(sql):
        r = subprocess.run(["beeline", "-u", url, "--silent=true", "--showHeader=true",
                            "--outputformat=csv2", "-e", sql + ";"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(r.stderr[-2000:])
        return r.stdout

    for nombre, sql in sentencias(HQL, parquet):
        t0 = time.time()
        salida = beeline("USE dncp; " + sql if not sql.upper().startswith(("CREATE DATABASE", "USE")) else sql)
        if nombre:
            df = pd.read_csv(io.StringIO(salida))
            df.columns = [c.split(".")[-1] for c in df.columns]
            comun.guardar_csv(df, "hive", nombre)
            print(f"   {time.time() - t0:.1f} s")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--beeline", help="URL JDBC de HiveServer2, p. ej. jdbc:hive2://")
    parser.add_argument("--parquet", default=comun.PARQUET, help="ruta de las tablas (p. ej. en HDFS)")
    args = parser.parse_args()
    if args.beeline:
        con_beeline(args.beeline, args.parquet)
    else:
        con_spark()

In [ ]:
%%writefile src/05_elasticsearch.py
"""Paso 5 (Elasticsearch): indexación y búsqueda para las preguntas 4 y 7.

  - Índice `dncp_contratos`: un documento por contrato y proveedor. P4: dado un RUC,
    sus contratos, entidades y montos, con el tiempo de respuesta en milisegundos.
  - Índice `dncp_items`: un documento por ítem adjudicado, con la descripción analizada
    (minúsculas y sin tildes). P7: gasto en ítems que mencionan combustible,
    medicamentos o alimentos, y entidades que concentran ese gasto.

Ambos índices tienen un único shard, así las agregaciones `terms` son exactas.

Uso:  python src/05_elasticsearch.py [--url http://localhost:9200] [--sin-carga]
"""
import argparse
import glob
import math
import time

import pandas as pd
import pyarrow.parquet as pq
from elasticsearch import Elasticsearch, helpers

import comun

IDX_CONTRATOS = f"dncp_contratos_{comun.ALCANCE}"
IDX_ITEMS = f"dncp_items_{comun.ALCANCE}"
PREFIJOS_UNSPSC = {"combustible": "1510", "medicamentos": "51", "alimentos": "50"}

AJUSTES = {
    "number_of_shards": 1,
    "number_of_replicas": 0,
    "refresh_interval": "-1",  # se reactiva al terminar la carga
    "analysis": {
        # El tokenizador estándar no corta "COMBUSTIBLE.MOVIL" ni "prov.combustibles":
        # punto, dos puntos y apóstrofos pasan a ser espacios antes de tokenizar.
        "char_filter": {"puntuacion_a_espacio": {"type": "mapping", "mappings": [
            ". => \\u0020", ": => \\u0020", "' => \\u0020", "’ => \\u0020"]}},
        "analyzer": {"es_sin_tildes": {
            "type": "custom", "char_filter": ["puntuacion_a_espacio"], "tokenizer": "standard",
            "filter": ["lowercase", "asciifolding"]}}},
}

MAPEO_CONTRATOS = {"properties": {
    "ruc": {"type": "keyword"}, "proveedor": {"type": "text", "fields": {"k": {"type": "keyword"}}},
    "ocid": {"type": "keyword"}, "contrato_id": {"type": "keyword"}, "fecha_firma": {"type": "date"},
    "entidad_id": {"type": "keyword"}, "entidad": {"type": "keyword"}, "tipo_entidad": {"type": "keyword"},
    "modalidad": {"type": "keyword"}, "categoria": {"type": "keyword"}, "anio": {"type": "integer"},
    "monto": {"type": "double"}, "moneda": {"type": "keyword"}}}

MAPEO_ITEMS = {"properties": {
    "descripcion": {"type": "text", "analyzer": "es_sin_tildes"},
    "catalogo": {"type": "text", "analyzer": "es_sin_tildes"},
    "catalogo_id": {"type": "keyword"}, "ocid": {"type": "keyword"}, "entidad": {"type": "keyword"},
    "anio": {"type": "integer"}, "monto": {"type": "double"}, "moneda": {"type": "keyword"}}}


def documentos(tabla, columnas, indice, filtro=None):
    for archivo in sorted(glob.glob(f"{comun.PARQUET_LOCAL}/{tabla}/*.parquet")):
        for lote in pq.ParquetFile(archivo).iter_batches(batch_size=20_000, columns=columnas):
            for fila in lote.to_pylist():
                if filtro and not filtro(fila):
                    continue
                doc = {k: v for k, v in fila.items()
                       if v is not None and not (isinstance(v, float) and math.isnan(v))}
                if "fecha_firma" in doc:
                    doc["fecha_firma"] = doc["fecha_firma"].isoformat()
                yield {"_index": indice, "_source": doc}


def cargar(es, indice, mapeo, acciones):
    if es.indices.exists(index=indice):
        es.indices.delete(index=indice)
    es.indices.create(index=indice, settings=AJUSTES, mappings=mapeo)
    t0 = time.time()
    ok = 0
    for exito, _ in helpers.parallel_bulk(es, acciones, thread_count=4, chunk_size=5000,
                                          request_timeout=300):
        ok += exito
    es.indices.put_settings(index=indice, settings={"refresh_interval": "1s"})
    es.indices.refresh(index=indice)
    es.indices.forcemerge(index=indice, max_num_segments=1, request_timeout=1800)
    print(f"  {indice}: {ok:,} documentos en {time.time() - t0:.0f} s")
    return ok, time.time() - t0


def p4(es):
    print("P4: consulta por RUC")
    muestra = pd.read_csv(f"{comun.RESULTADOS}/spark/p4_rucs_muestra.csv", dtype=str)["ruc"]
    filas = []
    for ruc in muestra:
        cuerpo = {
            "query": {"term": {"ruc": ruc}},
            "size": 0,
            "track_total_hits": True,
            "aggs": {"entidades": {"cardinality": {"field": "entidad_id", "precision_threshold": 40000}},
                     "pyg": {"filter": {"term": {"moneda": "PYG"}},
                             "aggs": {"monto": {"sum": {"field": "monto"}}}}},
        }
        t0 = time.perf_counter()
        r = es.search(index=IDX_CONTRATOS, body=cuerpo, request_cache=False)
        ms_total = 1000 * (time.perf_counter() - t0)
        filas.append({"ruc": ruc, "contratos": r["hits"]["total"]["value"],
                      "entidades": r["aggregations"]["entidades"]["value"],
                      "monto_pyg": r["aggregations"]["pyg"]["monto"]["value"],
                      "ms_es_servidor": r["took"], "ms_es_total": round(ms_total, 1)})
    comun.guardar_csv(pd.DataFrame(filas), "elasticsearch", "p4_consulta_ruc")

    # Listado completo para el RUC de ejemplo (el mismo que exporta Spark)
    ejemplo = muestra.iloc[0]
    t0 = time.perf_counter()
    docs = [h["_source"] for h in helpers.scan(es, index=IDX_CONTRATOS,
                                               query={"query": {"term": {"ruc": ejemplo}}})]
    print(f"  listado de {len(docs)} contratos del RUC {ejemplo}: "
          f"{1000 * (time.perf_counter() - t0):.0f} ms")
    df = pd.DataFrame(docs).sort_values("monto", ascending=False)
    comun.guardar_csv(df[["ruc", "proveedor", "ocid", "contrato_id", "fecha_firma", "entidad", "tipo_entidad",
                          "modalidad", "categoria", "monto", "moneda"]],
                      "elasticsearch", "p4_ejemplo_contratos_ruc")


def p7(es):
    print("P7: búsqueda de texto en ítems adjudicados")
    base = [{"term": {"moneda": "PYG"}}, {"exists": {"field": "monto"}}]
    aggs = {"monto": {"sum": {"field": "monto"}},
            "procesos": {"cardinality": {"field": "ocid", "precision_threshold": 40000}},
            "entidades": {"terms": {"field": "entidad", "size": 10,
                                    "order": [{"monto_entidad": "desc"}, {"_key": "asc"}]},
                          "aggs": {"monto_entidad": {"sum": {"field": "monto"}}}}}
    consultas = {
        "texto": {g: {"match": {"descripcion": {"query": " ".join(p), "operator": "or"}}}
                  for g, p in comun.TERMINOS_P7.items()},
        "catalogo": {g: {"prefix": {"catalogo_id": pref}} for g, pref in PREFIJOS_UNSPSC.items()},
    }
    for metodo, por_grupo in consultas.items():
        resumen, top = [], []
        for grupo, condicion in por_grupo.items():
            cuerpo = {"query": {"bool": {"filter": base, "must": [condicion]}}, "size": 0,
                      "track_total_hits": True, "aggs": aggs}
            r = es.search(index=IDX_ITEMS, body=cuerpo, request_cache=False)
            a = r["aggregations"]
            resumen.append({"grupo": grupo, "items": r["hits"]["total"]["value"],
                            "procesos": a["procesos"]["value"], "monto_pyg": a["monto"]["value"],
                            "ms_es_servidor": r["took"]})
            for b in a["entidades"]["buckets"]:
                top.append({"grupo": grupo, "entidad": b["key"], "monto_pyg": b["monto_entidad"]["value"],
                            "items": b["doc_count"]})
        comun.guardar_csv(pd.DataFrame(resumen), "elasticsearch", f"p7_{metodo}_resumen")
        comun.guardar_csv(pd.DataFrame(top), "elasticsearch", f"p7_{metodo}_top_entidades")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--url", default="http://localhost:9200")
    parser.add_argument("--sin-carga", action="store_true", help="usar los índices ya cargados")
    args = parser.parse_args()
    es = Elasticsearch(args.url, request_timeout=120)
    print(f"Elasticsearch {es.info()['version']['number']} en {args.url}")

    if not args.sin_carga:
        carga = []
        n, s = cargar(es, IDX_CONTRATOS, MAPEO_CONTRATOS, documentos(
            "contratos", ["ruc", "proveedor", "ocid", "contrato_id", "fecha_firma", "entidad_id", "entidad",
                          "tipo_entidad", "modalidad", "categoria", "anio", "monto", "moneda"], IDX_CONTRATOS))
        carga.append({"indice": IDX_CONTRATOS, "documentos": n, "segundos": round(s, 1)})
        n, s = cargar(es, IDX_ITEMS, MAPEO_ITEMS, documentos(
            "items_adjudicados", ["descripcion", "catalogo", "catalogo_id", "ocid", "entidad", "anio",
                                  "monto", "moneda"], IDX_ITEMS))
        carga.append({"indice": IDX_ITEMS, "documentos": n, "segundos": round(s, 1)})
        comun.guardar_csv(pd.DataFrame(carga), "elasticsearch", "00_carga_indices")
    p4(es)
    p7(es)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/05b_hbase.py
"""Paso 5b (Apache HBase): almacén clave-valor para la pregunta 4.

Tabla `dncp_contratos_por_proveedor`, familia de columnas `c`.
Clave de fila: <ruc>|<ocid>|<contrato_id>. Al estar ordenadas por clave, todas las
filas de un proveedor quedan contiguas y se leen con un scan por prefijo, sin
recorrer la tabla: la respuesta queda en milisegundos aunque haya millones de filas.

Requiere HBase con el servidor Thrift activo (en Colab: `hbase-daemon.sh start thrift`)
y la librería happybase.

Uso:  python src/05b_hbase.py [--host localhost] [--sin-carga]
"""
import argparse
import glob
import math
import time

import pandas as pd
import pyarrow.parquet as pq

import comun

TABLA = f"dncp_contratos_por_proveedor_{comun.ALCANCE}"
COLUMNAS = ["ruc", "proveedor", "ocid", "contrato_id", "fecha_firma", "entidad_id", "entidad",
            "tipo_entidad", "modalidad", "categoria", "anio", "monto", "moneda"]


def cargar(conexion):
    if TABLA.encode() in conexion.tables():
        conexion.delete_table(TABLA, disable=True)
    conexion.create_table(TABLA, {"c": dict(max_versions=1)})
    tabla = conexion.table(TABLA)
    t0, n = time.time(), 0
    with tabla.batch(batch_size=5000) as lote:
        for archivo in sorted(glob.glob(f"{comun.PARQUET_LOCAL}/contratos/*.parquet")):
            for bloque in pq.ParquetFile(archivo).iter_batches(batch_size=20_000, columns=COLUMNAS):
                for f in bloque.to_pylist():
                    if not f["ruc"]:
                        continue
                    clave = f"{f['ruc']}|{f['ocid']}|{f['contrato_id']}".encode()
                    datos = {f"c:{k}".encode(): str(v).encode() for k, v in f.items()
                             if v is not None and not (isinstance(v, float) and math.isnan(v))}
                    lote.put(clave, datos)
                    n += 1
    print(f"  {TABLA}: {n:,} filas en {time.time() - t0:.0f} s")
    return n, time.time() - t0


def consultar(tabla, ruc):
    contratos, entidades, monto = 0, set(), None
    for _, fila in tabla.scan(row_prefix=f"{ruc}|".encode(), batch_size=1000):
        contratos += 1
        if b"c:entidad_id" in fila:
            entidades.add(fila[b"c:entidad_id"])
        if fila.get(b"c:moneda") == b"PYG" and b"c:monto" in fila:
            monto = (monto or 0.0) + float(fila[b"c:monto"])
    return contratos, len(entidades), monto


def main():
    import happybase

    parser = argparse.ArgumentParser()
    parser.add_argument("--host", default="localhost")
    parser.add_argument("--sin-carga", action="store_true")
    args = parser.parse_args()
    conexion = happybase.Connection(args.host, timeout=600_000)
    if not args.sin_carga:
        n, s = cargar(conexion)
        comun.guardar_csv(pd.DataFrame([{"tabla": TABLA, "filas": n, "segundos": round(s, 1)}]),
                          "hbase", "00_carga_tabla")

    print("P4: consulta por RUC (scan por prefijo de clave)")
    tabla = conexion.table(TABLA)
    muestra = pd.read_csv(f"{comun.RESULTADOS}/spark/p4_rucs_muestra.csv", dtype=str)["ruc"]
    filas = []
    for ruc in muestra:
        t0 = time.perf_counter()
        contratos, entidades, monto = consultar(tabla, ruc)
        filas.append({"ruc": ruc, "contratos": contratos, "entidades": entidades, "monto_pyg": monto,
                      "ms_hbase": round(1000 * (time.perf_counter() - t0), 1)})
    comun.guardar_csv(pd.DataFrame(filas), "hbase", "p4_consulta_ruc")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/06_verificar.py
"""Paso 6: verificación cruzada de los exportables entre herramientas.

Para cada exportable se comparan las mismas filas (por claves) y columnas entre
Spark y las demás herramientas que lo calcularon:

    P1, P2      Spark  vs MapReduce (parseo independiente del JSON)  vs Hive
    P3, P5, P6  Spark  vs Hive
    P1, P3, P5  Spark  vs DuckDB      P2, P6  Spark  vs Polars        (clase 4)
    P4          Spark  vs Elasticsearch  (vs HBase si existe resultados/<alcance>/hbase)
    P6 Spearman Spark MLlib vs pandas (recalculado desde el Parquet)
    P7          Spark  vs Elasticsearch

Conteos: deben ser idénticos. Montos: diferencia relativa < 1e-9 (la suma en punto
flotante depende del orden). Porcentajes y percentiles: diferencia < 0.011 puntos
(Hive calcula el percentil del desvío en centésimas). Resultado:
resultados/<alcance>/verificacion.csv
"""
import glob
import os

import numpy as np
import pandas as pd

import comun

EXACTO, MONTO, PCT = "exacto", "monto", "pct"
TOL = {EXACTO: 0.0, MONTO: 1e-9, PCT: 0.011}


def leer(herramienta, nombre):
    ruta = os.path.join(comun.RESULTADOS, herramienta, f"{nombre}.csv")
    if not os.path.exists(ruta):
        return None
    return pd.read_csv(ruta, dtype=str, keep_default_na=False)


def comparar(pregunta, nombre, a, b, claves, columnas, tol_extra=None):
    """Compara la herramienta `a` (referencia) con `b` en las columnas dadas."""
    da, db = leer(a, nombre), leer(b, nombre)
    fila = {"pregunta": pregunta, "exportable": nombre, "referencia": a, "comparada_con": b}
    if da is None or db is None:
        return {**fila, "estado": "SIN DATOS", "detalle": f"falta el CSV de {a if da is None else b}"}
    if not claves:  # exportable de una sola fila
        da, db, claves = da.assign(_fila="1"), db.assign(_fila="1"), ["_fila"]
    unido = da.merge(db, on=claves, how="outer", suffixes=("_a", "_b"), indicator=True)
    solo = unido["_merge"] != "both"
    fila.update(filas_ref=len(da), filas_comp=len(db), claves_sin_par=int(solo.sum()))
    peores, max_dif = [], 0.0
    ambos = unido[~solo]
    for col, tipo in columnas.items():
        x = pd.to_numeric(ambos[f"{col}_a"], errors="coerce")
        y = pd.to_numeric(ambos[f"{col}_b"], errors="coerce")
        nulos_distintos = int((x.isna() != y.isna()).sum())
        if tipo == MONTO:
            dif = (x - y).abs() / np.maximum(x.abs(), 1.0)
        else:
            dif = (x - y).abs()
        tol = (tol_extra or {}).get(col, TOL[tipo])
        malos = int((dif > tol).sum()) + nulos_distintos
        d = float(dif.max()) if len(dif.dropna()) else 0.0
        max_dif = max(max_dif, d)
        if malos:
            peores.append(f"{col}: {malos} filas (máx. dif. {d:.4g})")
    ok = not peores and fila["claves_sin_par"] == 0
    fila.update(columnas=", ".join(columnas), max_dif=max_dif, estado="OK" if ok else "DIFERENCIA",
                detalle="; ".join(peores) or ("" if ok else f"{fila['claves_sin_par']} claves sin par"))
    return fila


def spearman_pandas():
    """Recalcula el rho de Spearman de P6 con pandas, sin Spark."""
    archivos = glob.glob(f"{comun.PARQUET_LOCAL}/procesos/*.parquet")
    df = pd.concat(pd.read_parquet(a, columns=["modalidad", "dias_convocatoria", "n_oferentes"])
                   for a in archivos)
    df = df[df["dias_convocatoria"].between(0, 365) & (df["n_oferentes"] >= 1)]
    spark = leer("spark", "p6_spearman")
    filas = []
    for _, r in spark.iterrows():
        d = df if r["modalidad"] == "(todas)" else df[df["modalidad"] == r["modalidad"]]
        filas.append({"modalidad": r["modalidad"], "llamados": len(d),
                      "spearman": round(d["dias_convocatoria"].corr(d["n_oferentes"], method="spearman"), 4)})
    comun.guardar_csv(pd.DataFrame(filas), "pandas", "p6_spearman")


def main():
    spearman_pandas()
    P1_COLS = {"procesos": EXACTO, "adjudicaciones": EXACTO, "contratos": EXACTO, "monto_adjudicado_pyg": MONTO}
    P2_COLS = {"llamados_con_oferta": EXACTO, "una_sola_oferta": EXACTO, "pct_una_oferta": PCT}
    checks = []
    for otra in ["mapreduce", "hive"]:
        checks.append(comparar("P1", "p1_anio_modalidad", "spark", otra, ["anio", "modalidad"], P1_COLS))
        for dim in ["modalidad", "tipo_entidad", "anio"]:
            checks.append(comparar("P2", f"p2_unica_oferta_{dim}", "spark", otra, [dim], P2_COLS))
    checks += [
        comparar("P3", "p3_desvio_resumen", "spark", "hive", [],
                 {"procesos": EXACTO, "pct_sobre_estimado": PCT, "pct_sobre_10": PCT, "pct_bajo_10": PCT,
                  "monto_estimado_pyg": MONTO, "monto_adjudicado_pyg": MONTO,
                  "p25": PCT, "mediana": PCT, "p75": PCT, "p90": PCT}),
        comparar("P3", "p3_desvio_categoria", "spark", "hive", ["categoria"],
                 {"procesos": EXACTO, "pct_sobre_10": PCT, "desvio_agregado_pct": PCT,
                  "mediana": PCT, "p90": PCT}),
        comparar("P5", "p5_concentracion_entidad", "spark", "hive", ["entidad_id"],
                 {"adjudicaciones": EXACTO, "proveedores": EXACTO, "monto_pyg": MONTO,
                  "share_top1": PCT, "share_top10": PCT, "hhi": PCT}, {"hhi": 0.011}),
        comparar("P5", "p5_entidades_por_nivel_hhi", "spark", "hive", ["nivel_hhi"],
                 {"entidades": EXACTO, "monto_pyg": MONTO}),
        comparar("P6", "p6_dias_modalidad", "spark", "hive", ["modalidad"],
                 {"procesos": EXACTO, "media": PCT, "p25": EXACTO, "mediana": EXACTO, "p75": EXACTO}),
        comparar("P6", "p6_tramos_dias_oferentes", "spark", "hive", ["tramo_dias"],
                 {"llamados": EXACTO, "oferentes_promedio": PCT, "oferentes_mediana": EXACTO,
                  "pct_una_oferta": PCT}),
        comparar("P6", "p6_spearman", "spark", "pandas", ["modalidad"],
                 {"llamados": EXACTO, "spearman": PCT}, {"spearman": 0.0001}),
    ]
    for otra in ["elasticsearch", "hbase"]:
        if otra == "hbase" and not os.path.isdir(os.path.join(comun.RESULTADOS, "hbase")):
            continue
        checks.append(comparar("P4", "p4_consulta_ruc", "spark", otra, ["ruc"],
                               {"contratos": EXACTO, "entidades": EXACTO, "monto_pyg": MONTO}))
    checks.append(comparar("P4", "p4_ejemplo_contratos_ruc", "spark", "elasticsearch",
                           ["ocid", "contrato_id"], {"monto": MONTO}))
    for metodo in ["texto", "catalogo"]:
        checks.append(comparar("P7", f"p7_{metodo}_resumen", "spark", "elasticsearch", ["grupo"],
                               {"items": EXACTO, "procesos": EXACTO, "monto_pyg": MONTO}))
        checks.append(comparar("P7", f"p7_{metodo}_top_entidades", "spark", "elasticsearch",
                               ["grupo", "entidad"], {"monto_pyg": MONTO, "items": EXACTO}))

    # DuckDB (SQL sobre Parquet) y Polars (lazy), clase 4
    checks.append(comparar("P1", "p1_anio_modalidad", "spark", "duckdb", ["anio", "modalidad"], P1_COLS))
    checks.append(comparar("P3", "p3_desvio_resumen", "spark", "duckdb", [],
                           {"procesos": EXACTO, "pct_sobre_estimado": PCT, "pct_sobre_10": PCT, "pct_bajo_10": PCT,
                            "monto_estimado_pyg": MONTO, "monto_adjudicado_pyg": MONTO,
                            "p25": PCT, "mediana": PCT, "p75": PCT, "p90": PCT}))
    checks.append(comparar("P5", "p5_concentracion_entidad", "spark", "duckdb", ["entidad_id"],
                           {"adjudicaciones": EXACTO, "proveedores": EXACTO, "monto_pyg": MONTO,
                            "share_top1": PCT, "share_top10": PCT, "hhi": PCT}))
    for dim in ["modalidad", "tipo_entidad", "anio"]:
        checks.append(comparar("P2", f"p2_unica_oferta_{dim}", "spark", "polars", [dim], P2_COLS))
    checks.append(comparar("P6", "p6_dias_modalidad", "spark", "polars", ["modalidad"],
                           {"procesos": EXACTO, "media": PCT, "p25": EXACTO, "mediana": EXACTO, "p75": EXACTO}))
    checks.append(comparar("P6", "p6_tramos_dias_oferentes", "spark", "polars", ["tramo_dias"],
                           {"llamados": EXACTO, "oferentes_promedio": PCT, "oferentes_mediana": EXACTO,
                            "pct_una_oferta": PCT}))

    df = pd.DataFrame(checks)
    ruta = os.path.join(comun.RESULTADOS, "verificacion.csv")
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    pd.set_option("display.width", 200)
    print(df[["pregunta", "exportable", "referencia", "comparada_con", "filas_ref", "filas_comp",
              "estado", "detalle"]].to_string(index=False))
    print(f"\n{(df['estado'] == 'OK').sum()} de {len(df)} comparaciones OK -> {os.path.relpath(ruta, comun.RAIZ)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/mapreduce/mapper.py
#!/usr/bin/env python3
"""Mapper de Hadoop Streaming para las preguntas 1 y 2.

Entrada: líneas del JSON Lines OCDS (un proceso por línea).
Salida:  clave<TAB>valores, con la clave compuesta por pregunta y dimensiones.

    P1  anio  modalidad      -> procesos, adjudicaciones, contratos, monto_adjudicado_pyg
    P2M modalidad            -> llamados_con_oferta, una_sola_oferta
    P2E tipo_entidad         -> llamados_con_oferta, una_sola_oferta
    P2A anio                 -> llamados_con_oferta, una_sola_oferta

Las definiciones replican las de src/01_aplanar_spark.py, pero el parseo del JSON
es independiente: si Spark y MapReduce coinciden, el aplanado es correcto.

El tipo de entidad faltante se completa con la tabla entidad -> tipo que genera el
trabajo 1 (mapper_entidades.py). Hadoop la distribuye a cada mapper con -files
(caché distribuida); el nombre del archivo puede pasarse como primer argumento.
"""
import os
import re
import sys

try:
    import orjson as _json
    cargar = _json.loads
except ImportError:  # en los nodos de Hadoop basta con la librería estándar
    import json
    cargar = json.loads

VALIDAS = ("active", "closed")
SIN_DATO = "(sin dato)"
# Clase \s de Java (solo ASCII), para coincidir con regexp_replace de Spark
ESPACIOS = re.compile(r"[ \t\n\x0b\f\r]+")


def limpiar(texto):
    """Espacios, tabuladores y saltos de línea seguidos -> un espacio."""
    return ESPACIOS.sub(" ", texto or "").strip(" ")


def numero(x):
    return float(x) if isinstance(x, (int, float)) and not isinstance(x, bool) else None


def cargar_tipos(ruta):
    tipos = {}
    if os.path.exists(ruta):
        with open(ruta, encoding="utf-8") as f:
            for linea in f:
                entidad_id, tipo = linea.rstrip("\r\n").split("\t", 1)
                tipos[entidad_id] = tipo
    return tipos


TIPOS = {}


def procesar(r):
    t = r.get("tender")
    if not t:
        return
    periodo = t.get("tenderPeriod") or {}
    fecha = periodo.get("startDate") or t.get("datePublished") or r.get("date") or ""
    anio = fecha[:4] if fecha[:4].isdigit() else SIN_DATO
    modalidad = limpiar(t.get("procurementMethodDetails")).lower() or SIN_DATO

    tipo_entidad = "Sin dato"
    for p in r.get("parties") or []:
        if "buyer" in (p.get("roles") or []):
            tipo_entidad = limpiar((p.get("details") or {}).get("level")) or "Sin dato"
            break
    entidad_id = (r.get("buyer") or {}).get("id")
    if entidad_id is not None:
        tipo_entidad = TIPOS.get(entidad_id, tipo_entidad)

    adjudicaciones, monto = 0, 0.0
    for a in r.get("awards") or []:
        if a.get("status") in VALIDAS:
            adjudicaciones += 1
            v = a.get("value") or {}
            if v.get("currency") == "PYG":
                monto += numero(v.get("amount")) or 0.0
    contratos = sum(1 for c in r.get("contracts") or [] if c.get("awardID") is not None)

    ids_bids = {x.get("id") for b in (r.get("bids") or {}).get("details") or []
                for x in b.get("tenderers") or []}
    n_oferentes = max(t.get("numberOfTenderers") or 0, len(t.get("tenderers") or []), len(ids_bids))

    yield f"P1\t{anio}\t{modalidad}", f"1\t{adjudicaciones}\t{contratos}\t{monto!r}"
    if n_oferentes >= 1:
        unica = 1 if n_oferentes == 1 else 0
        yield f"P2M\t{modalidad}", f"1\t{unica}"
        yield f"P2E\t{tipo_entidad}", f"1\t{unica}"
        yield f"P2A\t{anio}", f"1\t{unica}"


def main():
    TIPOS.update(cargar_tipos(sys.argv[1] if len(sys.argv) > 1 else "entidades_tipo.tsv"))
    salida = sys.stdout
    for linea in sys.stdin.buffer:
        linea = linea.strip()
        if not linea:
            continue
        try:
            registro = cargar(linea)
        except ValueError:
            salida.write("ERR\tjson_invalido\t1\n")
            continue
        for clave, valor in procesar(registro):
            salida.write(f"{clave}\t{valor}\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/mapreduce/reducer.py
#!/usr/bin/env python3
"""Reducer de Hadoop Streaming: suma los valores de cada clave.

Hadoop entrega las líneas ordenadas por clave; las claves son todos los campos menos
los valores numéricos del final. P1 lleva dos campos de clave y P2* uno.
"""
import sys

CAMPOS_CLAVE = {"P1": 3, "P2M": 2, "P2E": 2, "P2A": 2, "ERR": 2}


def separar(linea):
    partes = linea.rstrip("\n").split("\t")
    n = CAMPOS_CLAVE[partes[0]]
    return tuple(partes[:n]), partes[n:]


def emitir(clave, acumulado):
    valores = [repr(v) if isinstance(v, float) else str(v) for v in acumulado]
    sys.stdout.write("\t".join(list(clave) + valores) + "\n")


def main():
    actual, acumulado = None, None
    for linea in sys.stdin:
        clave, valores = separar(linea)
        numeros = [float(v) if "." in v or "e" in v else int(v) for v in valores]
        if clave != actual:
            if actual is not None:
                emitir(actual, acumulado)
            actual, acumulado = clave, numeros
        else:
            acumulado = [a + b for a, b in zip(acumulado, numeros)]
    if actual is not None:
        emitir(actual, acumulado)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/mapreduce/mapper_entidades.py
#!/usr/bin/env python3
"""Trabajo 1 de MapReduce: tipo de entidad informado por cada entidad compradora.

Emite entidad_id<TAB>tipo para cada proceso cuyo comprador trae el tipo en `parties`.
El reducer se queda con un valor por entidad y el resultado se distribuye al
trabajo 2 como archivo de caché (map-side join).
"""
import sys

from mapper import cargar, limpiar


def main():
    for linea in sys.stdin.buffer:
        linea = linea.strip()
        if not linea:
            continue
        try:
            r = cargar(linea)
        except ValueError:
            continue
        entidad_id = (r.get("buyer") or {}).get("id")
        if entidad_id is None or not r.get("tender"):
            continue
        for p in r.get("parties") or []:
            if "buyer" in (p.get("roles") or []):
                tipo = limpiar((p.get("details") or {}).get("level"))
                if tipo:
                    sys.stdout.write(f"{entidad_id}\t{tipo}\n")
                break


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/mapreduce/reducer_entidades.py
#!/usr/bin/env python3
"""Reducer del trabajo 1: un tipo por entidad (el mayor alfabéticamente, como MAX en SQL)."""
import sys


def main():
    actual, tipo = None, None
    for linea in sys.stdin:
        entidad_id, valor = linea.rstrip("\n").split("\t", 1)
        if entidad_id != actual:
            if actual is not None:
                sys.stdout.write(f"{actual}\t{tipo}\n")
            actual, tipo = entidad_id, valor
        else:
            tipo = max(tipo, valor)
    if actual is not None:
        sys.stdout.write(f"{actual}\t{tipo}\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/07_duckdb_polars.py
"""Paso 7 (DuckDB y Polars, clase 4; formatos, clase 3): otra comprobación en una sola máquina.

DuckDB (SQL sobre los archivos Parquet, sin servidor) y Polars (DataFrames con evaluación perezosa)
recalculan P1, P2, P3, P5 y P6 sobre las mismas tablas que generó Spark. Sus CSV se comparan con los de
Spark en src/06_verificar.py.

Además mide, como en el laboratorio de formatos de la clase 3, cuánto ocupa la tabla de ítems y cuánto
tarda la misma consulta en CSV, CSV comprimido con gzip y Parquet con Snappy y con Zstd.

Uso:  python src/07_duckdb_polars.py
"""
import glob
import os
import time

import duckdb
import pandas as pd
import polars as pl

import comun

SIN_DATO = "(sin dato)"


def archivos(tabla):
    return sorted(glob.glob(f"{comun.PARQUET_LOCAL}/{tabla}/*.parquet"))


def ruta_duckdb(tabla):
    return "[" + ", ".join(f"'{a}'" for a in archivos(tabla)) + "]"


def con_duckdb():
    con = duckdb.connect()
    for tabla in ["procesos", "adjudicaciones", "items_adjudicados"]:
        con.sql(f"CREATE VIEW {tabla} AS SELECT * FROM read_parquet({ruta_duckdb(tabla)}, union_by_name = true)")
    return con


def duckdb_preguntas(con):
    print("DuckDB: P1, P3 y P5")
    p1 = con.sql(f"""
        SELECT COALESCE(CAST(anio AS VARCHAR), '{SIN_DATO}') AS anio,
               COALESCE(modalidad, '{SIN_DATO}')             AS modalidad,
               COUNT(*)                                       AS procesos,
               SUM(n_adjudicaciones)                          AS adjudicaciones,
               SUM(n_contratos)                               AS contratos,
               SUM(monto_adjudicado_pyg)                      AS monto_adjudicado_pyg
        FROM procesos GROUP BY ALL ORDER BY ALL""").df()
    comun.guardar_csv(p1, "duckdb", "p1_anio_modalidad")

    p3 = con.sql("""
        WITH d AS (
            SELECT monto_estimado, monto_adjudicado_pyg,
                   100 * (monto_adjudicado_pyg - monto_estimado) / monto_estimado AS desvio
            FROM procesos
            WHERE moneda_estimado = 'PYG' AND monto_estimado > 0
              AND n_adjudicaciones_pyg > 0 AND monto_adjudicado_pyg > 0)
        SELECT COUNT(*) AS procesos,
               quantile_cont(desvio, 0.25) AS p25, quantile_cont(desvio, 0.5) AS mediana,
               quantile_cont(desvio, 0.75) AS p75, quantile_cont(desvio, 0.9) AS p90,
               ROUND(100 * AVG(CASE WHEN desvio > 0 THEN 1 ELSE 0 END), 2)   AS pct_sobre_estimado,
               ROUND(100 * AVG(CASE WHEN desvio > 10 THEN 1 ELSE 0 END), 2)  AS pct_sobre_10,
               ROUND(100 * AVG(CASE WHEN desvio < -10 THEN 1 ELSE 0 END), 2) AS pct_bajo_10,
               SUM(monto_estimado) AS monto_estimado_pyg, SUM(monto_adjudicado_pyg) AS monto_adjudicado_pyg
        FROM d""").df()
    comun.guardar_csv(p3, "duckdb", "p3_desvio_resumen")

    p5 = con.sql("""
        WITH por_proveedor AS (
            SELECT entidad_id, ruc, SUM(monto_atribuido) AS monto, COUNT(*) AS n
            FROM adjudicaciones
            WHERE valida AND moneda = 'PYG' AND monto_atribuido > 0 AND ruc IS NOT NULL AND entidad_id IS NOT NULL
            GROUP BY ALL),
        cuotas AS (
            SELECT *, 100 * monto / SUM(monto) OVER (PARTITION BY entidad_id) AS cuota,
                   ROW_NUMBER() OVER (PARTITION BY entidad_id ORDER BY monto DESC, ruc) AS puesto
            FROM por_proveedor)
        SELECT entidad_id, SUM(n) AS adjudicaciones, COUNT(*) AS proveedores, SUM(monto) AS monto_pyg,
               ROUND(SUM(CASE WHEN puesto = 1 THEN cuota END), 4)   AS share_top1,
               ROUND(SUM(CASE WHEN puesto <= 10 THEN cuota END), 4) AS share_top10,
               ROUND(SUM(cuota * cuota), 4)                         AS hhi
        FROM cuotas GROUP BY entidad_id ORDER BY monto_pyg DESC""").df()
    comun.guardar_csv(p5, "duckdb", "p5_concentracion_entidad")


def polars_preguntas():
    print("Polars (lazy): P2 y P6")
    procesos = pl.scan_parquet(archivos("procesos"))
    con_oferta = procesos.filter(pl.col("n_oferentes") >= 1)
    for dim in ["modalidad", "tipo_entidad", "anio"]:
        df = (con_oferta
              .group_by(pl.col(dim).cast(pl.Utf8).fill_null(SIN_DATO).alias(dim))
              .agg(pl.len().alias("llamados_con_oferta"),
                   (pl.col("n_oferentes") == 1).sum().alias("una_sola_oferta"))
              .with_columns((100 * pl.col("una_sola_oferta") / pl.col("llamados_con_oferta")).round(2)
                            .alias("pct_una_oferta"))
              .sort(dim).collect())
        comun.guardar_csv(df.to_pandas(), "polars", f"p2_unica_oferta_{dim}")

    validos = procesos.filter(pl.col("dias_convocatoria").is_between(0, 365))
    dias = pl.col("dias_convocatoria")
    p6 = (validos.group_by(pl.col("modalidad").fill_null(SIN_DATO))
          .agg(pl.len().alias("procesos"), dias.mean().round(2).alias("media"),
               dias.quantile(0.25, "linear").alias("p25"), dias.quantile(0.5, "linear").alias("mediana"),
               dias.quantile(0.75, "linear").alias("p75"))
          .filter(pl.col("procesos") >= 30).sort("procesos", descending=True).collect())
    comun.guardar_csv(p6.to_pandas(), "polars", "p6_dias_modalidad")

    tramo = (pl.when(dias <= 7).then(pl.lit("0-7")).when(dias <= 14).then(pl.lit("08-14"))
             .when(dias <= 21).then(pl.lit("15-21")).when(dias <= 30).then(pl.lit("22-30"))
             .when(dias <= 60).then(pl.lit("31-60")).otherwise(pl.lit("61-365")))
    tramos = (validos.filter(pl.col("n_oferentes") >= 1).with_columns(tramo.alias("tramo_dias"))
              .group_by("tramo_dias")
              .agg(pl.len().alias("llamados"), pl.col("n_oferentes").mean().round(4).alias("oferentes_promedio"),
                   pl.col("n_oferentes").median().alias("oferentes_mediana"),
                   ((pl.col("n_oferentes") == 1).mean() * 100).round(2).alias("pct_una_oferta"))
              .sort("tramo_dias").collect())
    comun.guardar_csv(tramos.to_pandas(), "polars", "p6_tramos_dias_oferentes")


def formatos(con):
    """Clase 3: la misma tabla en cuatro formatos; tamaño y tiempo de la misma consulta."""
    print("Formatos: CSV, CSV gzip, Parquet Snappy y Parquet Zstd")
    carpeta = os.path.join(comun.BASE, f"formatos_{comun.ALCANCE}")
    os.makedirs(carpeta, exist_ok=True)
    destinos = {
        "CSV": (f"{carpeta}/items.csv", "(FORMAT csv, HEADER true)"),
        "CSV gzip": (f"{carpeta}/items.csv.gz", "(FORMAT csv, HEADER true, COMPRESSION gzip)"),
        "Parquet Snappy": (f"{carpeta}/items_snappy.parquet", "(FORMAT parquet, COMPRESSION snappy)"),
        "Parquet Zstd": (f"{carpeta}/items_zstd.parquet", "(FORMAT parquet, COMPRESSION zstd)"),
    }
    filas = []
    for formato, (ruta, opciones) in destinos.items():
        t0 = time.perf_counter()
        con.sql(f"COPY (SELECT * FROM items_adjudicados) TO '{ruta}' {opciones}")
        escritura = time.perf_counter() - t0
        lector = f"read_csv('{ruta}')" if formato.startswith("CSV") else f"read_parquet('{ruta}')"
        consulta = f"SELECT anio, SUM(monto) AS monto FROM {lector} GROUP BY anio"
        tiempos = []
        for _ in range(3):
            t0 = time.perf_counter()
            con.sql(consulta).fetchall()
            tiempos.append(time.perf_counter() - t0)
        filas.append({"formato": formato, "mb": round(os.path.getsize(ruta) / 1024**2, 1),
                      "escritura_s": round(escritura, 2), "consulta_monto_por_anio_s": round(min(tiempos), 3)})
    df = pd.DataFrame(filas)
    df["veces_mas_chico_que_csv"] = (df["mb"].iloc[0] / df["mb"]).round(1)
    comun.guardar_csv(df, "formatos", "comparacion_formatos_items")
    print(df.to_string(index=False))


if __name__ == "__main__":
    con = con_duckdb()
    duckdb_preguntas(con)
    polars_preguntas()
    formatos(con)

In [ ]:
%%writefile src/hive/preguntas.hql
-- =====================================================================
-- Apache Hive: tablas externas sobre el Parquet generado por Spark y
-- consultas HiveQL para las preguntas 1, 2, 3, 5 y 6.
--
-- Cada consulta marcada con "-- exportable: <nombre>" se guarda como
-- resultados/<alcance>/hive/<nombre>.csv (ver src/04_hive.py).
-- ${PARQUET} se reemplaza por la ruta de las tablas (HDFS en Colab).
-- Solo se usan funciones disponibles en Hive 3.x y en Spark SQL.
-- =====================================================================

CREATE DATABASE IF NOT EXISTS dncp;
USE dncp;

DROP TABLE IF EXISTS procesos;
CREATE EXTERNAL TABLE procesos (
    ocid STRING, anio INT, fecha_inicio DATE, fecha_fin DATE, dias_convocatoria INT,
    modalidad STRING, metodo STRING, categoria STRING, entidad_id STRING, entidad STRING,
    tipo_entidad STRING, estado_llamado STRING, moneda_estimado STRING, monto_estimado DOUBLE,
    n_oferentes BIGINT, n_adjudicaciones INT, n_contratos INT, monto_adjudicado_pyg DOUBLE,
    n_adjudicaciones_pyg INT)
STORED AS PARQUET LOCATION '${PARQUET}/procesos';

DROP TABLE IF EXISTS adjudicaciones;
CREATE EXTERNAL TABLE adjudicaciones (
    ocid STRING, anio INT, modalidad STRING, categoria STRING, entidad_id STRING, entidad STRING,
    tipo_entidad STRING, adjudicacion_id STRING, estado STRING, valida BOOLEAN, fecha DATE,
    monto DOUBLE, moneda STRING, n_proveedores INT, ruc STRING, proveedor STRING,
    monto_atribuido DOUBLE)
STORED AS PARQUET LOCATION '${PARQUET}/adjudicaciones';

-- ---------------------------------------------------------------------
-- P1. Procesos, adjudicaciones, contratos y monto adjudicado por año y modalidad
-- ---------------------------------------------------------------------
-- exportable: p1_anio_modalidad
SELECT COALESCE(CAST(anio AS STRING), '(sin dato)') AS anio,
       COALESCE(modalidad, '(sin dato)')            AS modalidad,
       COUNT(*)                                      AS procesos,
       SUM(n_adjudicaciones)                         AS adjudicaciones,
       SUM(n_contratos)                              AS contratos,
       SUM(monto_adjudicado_pyg)                     AS monto_adjudicado_pyg
FROM procesos
GROUP BY COALESCE(CAST(anio AS STRING), '(sin dato)'), COALESCE(modalidad, '(sin dato)')
ORDER BY anio, modalidad;

-- ---------------------------------------------------------------------
-- P2. Proporción de llamados con una sola oferta
-- ---------------------------------------------------------------------
-- exportable: p2_unica_oferta_modalidad
SELECT COALESCE(modalidad, '(sin dato)') AS modalidad,
       COUNT(*) AS llamados_con_oferta,
       SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) AS una_sola_oferta,
       ROUND(100 * SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_una_oferta
FROM procesos
WHERE n_oferentes >= 1
GROUP BY COALESCE(modalidad, '(sin dato)')
ORDER BY modalidad;

-- exportable: p2_unica_oferta_tipo_entidad
SELECT tipo_entidad,
       COUNT(*) AS llamados_con_oferta,
       SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) AS una_sola_oferta,
       ROUND(100 * SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_una_oferta
FROM procesos
WHERE n_oferentes >= 1
GROUP BY tipo_entidad
ORDER BY tipo_entidad;

-- exportable: p2_unica_oferta_anio
SELECT COALESCE(CAST(anio AS STRING), '(sin dato)') AS anio,
       COUNT(*) AS llamados_con_oferta,
       SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) AS una_sola_oferta,
       ROUND(100 * SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_una_oferta
FROM procesos
WHERE n_oferentes >= 1
GROUP BY COALESCE(CAST(anio AS STRING), '(sin dato)')
ORDER BY anio;

-- ---------------------------------------------------------------------
-- P3. Desvío porcentual entre monto estimado y adjudicado (procesos en PYG)
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS desvios;
CREATE VIEW desvios AS
SELECT COALESCE(categoria, '(sin dato)') AS categoria,
       COALESCE(modalidad, '(sin dato)') AS modalidad,
       monto_estimado, monto_adjudicado_pyg,
       100 * (monto_adjudicado_pyg - monto_estimado) / monto_estimado AS desvio_pct,
       -- PERCENTILE de Hive es exacto pero solo acepta enteros: desvío en centésimas de punto
       CAST(ROUND(10000 * (monto_adjudicado_pyg - monto_estimado) / monto_estimado) AS BIGINT)
           AS desvio_centesimas
FROM procesos
WHERE moneda_estimado = 'PYG' AND monto_estimado > 0
  AND n_adjudicaciones_pyg > 0 AND monto_adjudicado_pyg > 0;

-- exportable: p3_desvio_resumen
SELECT COUNT(*) AS procesos,
       PERCENTILE(desvio_centesimas, 0.25) / 100 AS p25,
       PERCENTILE(desvio_centesimas, 0.5) / 100  AS mediana,
       PERCENTILE(desvio_centesimas, 0.75) / 100 AS p75,
       PERCENTILE(desvio_centesimas, 0.9) / 100  AS p90,
       ROUND(100 * AVG(CASE WHEN desvio_pct > 0 THEN 1 ELSE 0 END), 2)   AS pct_sobre_estimado,
       ROUND(100 * AVG(CASE WHEN desvio_pct > 10 THEN 1 ELSE 0 END), 2)  AS pct_sobre_10,
       ROUND(100 * AVG(CASE WHEN desvio_pct < -10 THEN 1 ELSE 0 END), 2) AS pct_bajo_10,
       SUM(monto_estimado)       AS monto_estimado_pyg,
       SUM(monto_adjudicado_pyg) AS monto_adjudicado_pyg
FROM desvios;

-- exportable: p3_desvio_categoria
SELECT categoria,
       COUNT(*) AS procesos,
       PERCENTILE(desvio_centesimas, 0.5) / 100 AS mediana,
       PERCENTILE(desvio_centesimas, 0.9) / 100 AS p90,
       ROUND(100 * AVG(CASE WHEN desvio_pct > 0 THEN 1 ELSE 0 END), 2)  AS pct_sobre_estimado,
       ROUND(100 * AVG(CASE WHEN desvio_pct > 10 THEN 1 ELSE 0 END), 2) AS pct_sobre_10,
       SUM(monto_estimado)       AS monto_estimado_pyg,
       SUM(monto_adjudicado_pyg) AS monto_adjudicado_pyg,
       ROUND(100 * (SUM(monto_adjudicado_pyg) - SUM(monto_estimado)) / SUM(monto_estimado), 2)
           AS desvio_agregado_pct
FROM desvios
GROUP BY categoria
HAVING COUNT(*) >= 30
ORDER BY pct_sobre_10 DESC;

-- ---------------------------------------------------------------------
-- P5. Concentración: participación del top 10 e índice Herfindahl-Hirschman
-- ---------------------------------------------------------------------
DROP VIEW IF EXISTS cuotas_entidad;
CREATE VIEW cuotas_entidad AS
SELECT entidad_id, ruc, monto, n,
       100 * monto / SUM(monto) OVER (PARTITION BY entidad_id) AS share,
       ROW_NUMBER() OVER (PARTITION BY entidad_id ORDER BY monto DESC, ruc) AS rango
FROM (
    SELECT entidad_id, ruc, SUM(monto_atribuido) AS monto, COUNT(*) AS n
    FROM adjudicaciones
    WHERE valida AND moneda = 'PYG' AND monto_atribuido > 0
      AND ruc IS NOT NULL AND entidad_id IS NOT NULL
    GROUP BY entidad_id, ruc
) por_proveedor;

-- Nombre y tipo de cada entidad (el nombre cambia entre años; se toma uno fijo)
DROP VIEW IF EXISTS entidades;
CREATE VIEW entidades AS
SELECT entidad_id, MAX(entidad) AS entidad, MAX(tipo_entidad) AS tipo_entidad
FROM adjudicaciones
WHERE valida AND moneda = 'PYG' AND monto_atribuido > 0
  AND ruc IS NOT NULL AND entidad_id IS NOT NULL
GROUP BY entidad_id;

-- exportable: p5_concentracion_entidad
SELECT c.entidad_id, e.entidad, e.tipo_entidad, c.adjudicaciones, c.proveedores, c.monto_pyg,
       c.share_top1, c.share_top10, c.hhi
FROM (
    SELECT entidad_id,
           SUM(n) AS adjudicaciones,
           COUNT(*) AS proveedores,
           SUM(monto) AS monto_pyg,
           ROUND(SUM(CASE WHEN rango = 1 THEN share END), 4)   AS share_top1,
           ROUND(SUM(CASE WHEN rango <= 10 THEN share END), 4) AS share_top10,
           ROUND(SUM(share * share), 4) AS hhi
    FROM cuotas_entidad
    GROUP BY entidad_id
) c
JOIN entidades e ON c.entidad_id = e.entidad_id
ORDER BY monto_pyg DESC;

-- exportable: p5_entidades_por_nivel_hhi
SELECT nivel_hhi, COUNT(*) AS entidades, SUM(monto_pyg) AS monto_pyg
FROM (
    SELECT CASE WHEN hhi < 1500 THEN '1. no concentrado (<1500)'
                WHEN hhi <= 2500 THEN '2. moderado (1500-2500)'
                ELSE '3. alto (>2500)' END AS nivel_hhi,
           monto_pyg
    FROM (
        SELECT SUM(n) AS adjudicaciones, SUM(monto) AS monto_pyg, SUM(share * share) AS hhi
        FROM cuotas_entidad
        GROUP BY entidad_id
    ) e
    WHERE adjudicaciones >= 20
) c
GROUP BY nivel_hhi
ORDER BY nivel_hhi;

-- ---------------------------------------------------------------------
-- P6. Duración de la convocatoria y cantidad de oferentes
-- ---------------------------------------------------------------------
-- exportable: p6_dias_modalidad
SELECT COALESCE(modalidad, '(sin dato)') AS modalidad,
       COUNT(*) AS procesos,
       ROUND(AVG(dias_convocatoria), 2) AS media,
       PERCENTILE(CAST(dias_convocatoria AS BIGINT), 0.25) AS p25,
       PERCENTILE(CAST(dias_convocatoria AS BIGINT), 0.5)  AS mediana,
       PERCENTILE(CAST(dias_convocatoria AS BIGINT), 0.75) AS p75
FROM procesos
WHERE dias_convocatoria BETWEEN 0 AND 365
GROUP BY COALESCE(modalidad, '(sin dato)')
HAVING COUNT(*) >= 30
ORDER BY procesos DESC;

-- exportable: p6_tramos_dias_oferentes
SELECT tramo_dias,
       COUNT(*) AS llamados,
       ROUND(AVG(n_oferentes), 4) AS oferentes_promedio,
       PERCENTILE(n_oferentes, 0.5) AS oferentes_mediana,
       ROUND(100 * AVG(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END), 2) AS pct_una_oferta
FROM (
    SELECT n_oferentes,
           CASE WHEN dias_convocatoria <= 7  THEN '0-7'
                WHEN dias_convocatoria <= 14 THEN '08-14'
                WHEN dias_convocatoria <= 21 THEN '15-21'
                WHEN dias_convocatoria <= 30 THEN '22-30'
                WHEN dias_convocatoria <= 60 THEN '31-60'
                ELSE '61-365' END AS tramo_dias
    FROM procesos
    WHERE dias_convocatoria BETWEEN 0 AND 365 AND n_oferentes >= 1
) t
GROUP BY tramo_dias
ORDER BY tramo_dias;

## 2. Instalación: Java 8, Hadoop 3.3.6, Hive 3.1.3, HBase 2.5, Spark 3.5 y Elasticsearch 9

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y openjdk-8-jdk-headless > /dev/null
pip -q install "pyspark==3.5.*" "elasticsearch==9.*" happybase orjson pyarrow duckdb polars
cd /opt
A=https://archive.apache.org/dist
[ -d hadoop ] || { wget -q $A/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz && tar xzf hadoop-3.3.6.tar.gz && mv hadoop-3.3.6 hadoop; }
[ -d hive ]   || { wget -q $A/hive/hive-3.1.3/apache-hive-3.1.3-bin.tar.gz && tar xzf apache-hive-3.1.3-bin.tar.gz && mv apache-hive-3.1.3-bin hive; }
[ -d hbase ]  || { wget -q $A/hbase/2.5.10/hbase-2.5.10-hadoop3-bin.tar.gz && tar xzf hbase-2.5.10-hadoop3-bin.tar.gz && mv hbase-2.5.10-hadoop3 hbase; }
[ -d elasticsearch ] || { wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-9.5.1-linux-x86_64.tar.gz && tar xzf elasticsearch-9.5.1-linux-x86_64.tar.gz && mv elasticsearch-9.5.1 elasticsearch; }
rm -f *.tar.gz
ls /opt

### 2.1 Hadoop pseudodistribuido (HDFS) y MapReduce

In [ ]:
%%bash
set -e
H=/opt/hadoop/etc/hadoop
echo "export JAVA_HOME=$JAVA_HOME" >> $H/hadoop-env.sh
cat > $H/core-site.xml <<'X'
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
  <property><name>hadoop.proxyuser.root.hosts</name><value>*</value></property>
  <property><name>hadoop.proxyuser.root.groups</name><value>*</value></property>
</configuration>
X
cat > $H/hdfs-site.xml <<'X'
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>/content/hdfs/nn</value></property>
  <property><name>dfs.datanode.data.dir</name><value>/content/hdfs/dn</value></property>
</configuration>
X
# MapReduce con el ejecutor local de Hadoop (sin YARN, para no agotar la RAM de Colab)
cat > $H/mapred-site.xml <<'X'
<configuration>
  <property><name>mapreduce.framework.name</name><value>local</value></property>
  <property><name>mapreduce.task.io.sort.mb</name><value>256</value></property>
</configuration>
X
[ -d /content/hdfs/nn ] || hdfs namenode -format -nonInteractive -force > /dev/null 2>&1
hdfs --daemon start namenode
hdfs --daemon start datanode
sleep 10
hdfs dfs -mkdir -p /dncp /tmp /user/hive/warehouse
hdfs dfs -chmod -R 777 /tmp /user/hive
hdfs dfsadmin -report | head -12

### 2.2 Hive con metastore Derby

In [ ]:
%%bash
set -e
cd /opt/hive
rm -f lib/guava-19.0.jar && cp /opt/hadoop/share/hadoop/common/lib/guava-27.0-jre.jar lib/
cat > conf/hive-site.xml <<'X'
<configuration>
  <property><name>javax.jdo.option.ConnectionURL</name>
            <value>jdbc:derby:;databaseName=/content/metastore_db;create=true</value></property>
  <property><name>hive.metastore.warehouse.dir</name><value>/user/hive/warehouse</value></property>
  <property><name>hive.server2.enable.doAs</name><value>false</value></property>
  <property><name>hive.exec.mode.local.auto</name><value>true</value></property>
  <property><name>hive.metastore.schema.verification</name><value>false</value></property>
</configuration>
X
[ -d /content/metastore_db ] || bin/schematool -dbType derby -initSchema > /tmp/schematool.log 2>&1
tail -2 /tmp/schematool.log
beeline -u jdbc:hive2:// --silent=true -e "SELECT 'hive ok';" 2>/dev/null

### 2.3 HBase (modo standalone) con servidor Thrift

In [ ]:
%%bash
set -e
cd /opt/hbase
echo "export JAVA_HOME=$JAVA_HOME" >> conf/hbase-env.sh
cat > conf/hbase-site.xml <<'X'
<configuration>
  <property><name>hbase.cluster.distributed</name><value>false</value></property>
  <property><name>hbase.rootdir</name><value>file:///content/hbase</value></property>
  <property><name>hbase.zookeeper.property.dataDir</name><value>/content/zookeeper</value></property>
  <property><name>hbase.unsafe.stream.capability.enforce</name><value>false</value></property>
</configuration>
X
bin/start-hbase.sh
sleep 20
bin/hbase-daemon.sh start thrift
sleep 10
echo "status" | bin/hbase shell -n 2>/dev/null | tail -2

### 2.4 Elasticsearch (nodo único; no puede ejecutarse como root)

In [ ]:
%%bash
set -e
id es >/dev/null 2>&1 || useradd -m es
cd /opt/elasticsearch
cat >> config/elasticsearch.yml <<'X'
discovery.type: single-node
network.host: 127.0.0.1
xpack.security.enabled: false
xpack.ml.enabled: false
X
printf -- "-Xms2g\n-Xmx2g\n" > config/jvm.options.d/heap.options
chown -R es:es /opt/elasticsearch
sudo -u es bin/elasticsearch -d -p /tmp/es.pid
for i in $(seq 1 60); do curl -s localhost:9200 >/dev/null && break; sleep 2; done
curl -s localhost:9200 | grep -E '"number"|tagline'

## 3. Datos: descarga desde el OCP Data Registry y carga en HDFS

In [ ]:
%%bash
set -e
cd /content/dncp/datos
URL="https://data.open-contracting.org/en/publication/63/download?name=$ALCANCE.jsonl.gz"
[ -f $ALCANCE.jsonl.gz ] || wget -q -O $ALCANCE.jsonl.gz "$URL"
ls -lh $ALCANCE.jsonl.gz
# Se descomprime al cargar: un .gz no se puede dividir entre tareas de MapReduce/Spark
zcat $ALCANCE.jsonl.gz | hdfs dfs -put -f - /dncp/$ALCANCE.jsonl
hdfs dfs -ls -h /dncp
hdfs fsck /dncp/$ALCANCE.jsonl -blocks | grep -E "Total blocks|Total size"

## 4. Spark: aplanado del JSON OCDS a tablas Parquet en HDFS

In [ ]:
!python src/01_aplanar_spark.py 2>&1 | grep -vE "WARN|^\s*$"
!rm -rf $DNCP_PARQUET_LOCAL && hdfs dfs -get $DNCP_PARQUET $DNCP_PARQUET_LOCAL && du -sh $DNCP_PARQUET_LOCAL/*

## 5. Spark: las 7 preguntas

In [ ]:
!python src/02_preguntas_spark.py 2>&1 | grep -vE "WARN|^\\s*$"

## 6. MapReduce (Hadoop Streaming): preguntas 1 y 2
El mapper lee el JSON crudo desde HDFS, sin pasar por Spark: es una verificación independiente del aplanado.
Son dos trabajos: el primero arma la tabla entidad → tipo de entidad; el segundo la recibe por caché
distribuida (`-files`) para completar el tipo faltante (map-side join) y calcula P1 y P2. Como en el
laboratorio de la clase 3, el reducer también se usa como *combiner* (sumar es asociativo): los contadores
*Map output records* y *Combine output records* muestran cuántos registros dejan de viajar al reduce.

In [ ]:
%%bash
set -e
STREAMING=/opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar
FILTRO="map [0-9]+% reduce|Job .* completed|Map input records|Map output records|Combine output records|Reduce output records"
hdfs dfs -rm -r -f /dncp/salida_entidades /dncp/salida_mr

# Trabajo 1: entidad -> tipo de entidad
hadoop jar $STREAMING -D mapreduce.job.name="dncp_entidades" \
    -files src/mapreduce/mapper.py,src/mapreduce/mapper_entidades.py,src/mapreduce/reducer_entidades.py \
    -mapper "python3 mapper_entidades.py" -combiner "python3 reducer_entidades.py" -reducer "python3 reducer_entidades.py" \
    -input /dncp/$ALCANCE.jsonl -output /dncp/salida_entidades 2>&1 | grep -E "$FILTRO"
hdfs dfs -getmerge /dncp/salida_entidades /content/dncp/entidades_tipo.tsv
wc -l /content/dncp/entidades_tipo.tsv

# Trabajo 2: P1 y P2, con la tabla anterior en la caché distribuida
hadoop jar $STREAMING -D mapreduce.job.name="dncp_p1_p2" \
    -files src/mapreduce/mapper.py,src/mapreduce/reducer.py,/content/dncp/entidades_tipo.tsv \
    -mapper "python3 mapper.py entidades_tipo.tsv" -combiner "python3 reducer.py" -reducer "python3 reducer.py" \
    -input /dncp/$ALCANCE.jsonl -output /dncp/salida_mr 2>&1 | grep -E "$FILTRO"
hdfs dfs -getmerge /dncp/salida_mr /content/dncp/salida_mr.txt
python src/03_mapreduce.py --hadoop /content/dncp/salida_mr.txt

## 7. Hive: tablas externas sobre el Parquet en HDFS y consultas HiveQL

In [ ]:
!python src/04_hive.py --beeline "jdbc:hive2://" --parquet "$DNCP_PARQUET"

## 8. Elasticsearch: índices de contratos e ítems (P4 y P7)

In [ ]:
!python src/05_elasticsearch.py

## 9. HBase: contratos por proveedor con clave RUC (P4)

In [ ]:
!python src/05b_hbase.py

## 10. DuckDB y Polars (clase 4) y formatos (clase 3)

In [ ]:
!python src/07_duckdb_polars.py

## 11. Verificación cruzada de todos los exportables

In [ ]:
!python src/06_verificar.py
import pandas as pd
v = pd.read_csv(f"resultados/{ALCANCE}/verificacion.csv")
v[["pregunta", "exportable", "referencia", "comparada_con", "estado", "detalle"]]

## 12. Descargar los resultados

In [ ]:
!cd resultados && zip -qr /content/resultados_{ALCANCE}.zip {ALCANCE}
from google.colab import files
files.download(f"/content/resultados_{ALCANCE}.zip")